<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/train_class_prompt_labeling_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# 1 — GOOGLE DRIVE PERSISTENT STORAGE SETUP
# ROUTER AUTO TUNE V2

from google.colab import drive

import os
import shutil


# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

DRIVE_MOUNT = "/content/drive"

drive_ok = False

try:

    drive.mount(
        DRIVE_MOUNT,
        force_remount=False
    )

    assert os.path.exists(
        f"{DRIVE_MOUNT}/MyDrive"
    ), "Google Drive belum mounted"

    drive_ok = True

except Exception as e:

    print()
    print("WARNING: Google Drive gagal dimount.")
    print(type(e).__name__, str(e))

    drive_ok = False


# =========================================================
# 2. STORAGE DIRECTORY
# =========================================================

if drive_ok:

    SAVE_DIR = (
        f"{DRIVE_MOUNT}/MyDrive/router_classifier"
    )

else:

    print()
    print(
        "Fallback sementara ke /content."
    )

    print(
        "WARNING: data bisa hilang "
        "jika runtime restart."
    )

    SAVE_DIR = (
        "/content/router_classifier_temp"
    )


os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


# =========================================================
# 3. V1 BASELINE PATHS
#
# File lama hanya dibaca sebagai baseline.
# Jangan overwrite dari flow V2.
# =========================================================

BASE_PATH = (
    f"{SAVE_DIR}/router_teacher_checkpoint.csv"
)

EXTRA_ALL_PATH = (
    f"{SAVE_DIR}/router_teacher_extra_all.csv"
)

CURRENT_DATASET_PATH = (
    f"{SAVE_DIR}/router_dataset_current.csv"
)

V1_MODEL_PATH = (
    f"{SAVE_DIR}/router_classifier_current.joblib"
)

V1_ROUND_LOG_PATH = (
    f"{SAVE_DIR}/router_tuning_rounds.csv"
)


# =========================================================
# 4. ROUTER V2 PATHS
# =========================================================

V2_ACCEPTED_PATH = (
    f"{SAVE_DIR}/router_v2_accepted.csv"
)

V2_REJECTED_PATH = (
    f"{SAVE_DIR}/router_v2_rejected.csv"
)

V2_HISTORY_PATH = (
    f"{SAVE_DIR}/router_v2_history.csv"
)

V2_VALIDATION_PATH = (
    f"{SAVE_DIR}/router_v2_validation_fixed.csv"
)

V2_CURRENT_MODEL_PATH = (
    f"{SAVE_DIR}/router_v2_current.joblib"
)

V2_BEST_MODEL_PATH = (
    f"{SAVE_DIR}/router_v2_best.joblib"
)

V2_BEST_METRICS_PATH = (
    f"{SAVE_DIR}/router_v2_best_metrics.json"
)

V2_ACTIVE_DATASET_PATH = (
    f"{SAVE_DIR}/router_v2_active_dataset.csv"
)


# =========================================================
# 5. STORAGE STATUS
# =========================================================

print()
print("=" * 65)
print("ROUTER V2 STORAGE STATUS")
print("=" * 65)

print(
    "Drive mounted :",
    drive_ok
)

print(
    "SAVE_DIR      :",
    SAVE_DIR
)

print()

print(
    "V1 baseline dataset :",
    CURRENT_DATASET_PATH
)

print(
    "V2 accepted data    :",
    V2_ACCEPTED_PATH
)

print(
    "V2 rejected data    :",
    V2_REJECTED_PATH
)

print(
    "V2 validation       :",
    V2_VALIDATION_PATH
)

print(
    "V2 best model       :",
    V2_BEST_MODEL_PATH
)

print("=" * 65)

Mounted at /content/drive

ROUTER V2 STORAGE STATUS
Drive mounted : True
SAVE_DIR      : /content/drive/MyDrive/router_classifier

V1 baseline dataset : /content/drive/MyDrive/router_classifier/router_dataset_current.csv
V2 accepted data    : /content/drive/MyDrive/router_classifier/router_v2_accepted.csv
V2 rejected data    : /content/drive/MyDrive/router_classifier/router_v2_rejected.csv
V2 validation       : /content/drive/MyDrive/router_classifier/router_v2_validation_fixed.csv
V2 best model       : /content/drive/MyDrive/router_classifier/router_v2_best.joblib


In [2]:
# =========================================================
# CELL 2 — DEPENDENCIES + IMPORTS
# ROUTER AUTO TUNE V2
# =========================================================

import os
import re
import json
import time
import random
import joblib

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


# =========================================================
# RANDOM SEED
# =========================================================

RANDOM_STATE = 42

random.seed(
    RANDOM_STATE
)

np.random.seed(
    RANDOM_STATE
)


# =========================================================
# GLOBAL CONFIG
# =========================================================

MIN_CONFIDENCE = 0.80

MIN_IMPROVEMENT = 0.005

MAX_CLASS_RECALL_DROP = 0.05

CANDIDATE_BATCH_SIZE = 6

MAX_NO_IMPROVE_ROUNDS = 4


GRID_C_VALUES = [
    0.05,
    0.10,
    0.30,
    0.50,
    1.00,
    2.00,
    3.00,
    5.00,
    10.00,
]


# =========================================================
# CHECK
# =========================================================

print()
print("=" * 65)
print("ROUTER V2 ENVIRONMENT")
print("=" * 65)

print(
    "RANDOM_STATE          :",
    RANDOM_STATE
)

print(
    "MIN_CONFIDENCE        :",
    MIN_CONFIDENCE
)

print(
    "MIN_IMPROVEMENT       :",
    MIN_IMPROVEMENT
)

print(
    "CANDIDATE_BATCH_SIZE  :",
    CANDIDATE_BATCH_SIZE
)

print(
    "MAX_NO_IMPROVE_ROUNDS:",
    MAX_NO_IMPROVE_ROUNDS
)

print("=" * 65)


ROUTER V2 ENVIRONMENT
RANDOM_STATE          : 42
MIN_CONFIDENCE        : 0.8
MIN_IMPROVEMENT       : 0.005
CANDIDATE_BATCH_SIZE  : 6
MAX_NO_IMPROVE_ROUNDS: 4


In [3]:
# =========================================================
# 3 — ROUTING LABEL DEFINITIONS
# ROUTER AUTO TUNE V2
# =========================================================

LABELS = [
    "SIMPLE",
    "GENERAL",
    "REASONING",
    "CODING_SIMPLE",
    "CODING_COMPLEX",
    "TRANSFORM",
    "CREATIVE",
]


CATEGORY_DESCRIPTIONS = {

    "SIMPLE":
        "Simple factual questions, definitions, easy lookup, "
        "basic calculations, or very short tasks requiring "
        "little to no reasoning.",

    "GENERAL":
        "Normal assistant questions and explanations that "
        "require some understanding but do not require "
        "substantial multi-step reasoning, comparison, "
        "planning, or software engineering analysis.",

    "REASONING":
        "Non-coding analytical tasks requiring comparison, "
        "planning, trade-offs, deduction, mathematics, "
        "decision making, or multi-step reasoning. "
        "Technology comparisons without actual software "
        "debugging or implementation belong here.",

    "CODING_SIMPLE":
        "Small programming tasks such as regex, SQL, syntax "
        "correction, short scripts, small functions, simple "
        "code modification, or straightforward implementation.",

    "CODING_COMPLEX":
        "Software engineering tasks requiring substantial "
        "technical reasoning involving debugging, architecture, "
        "security, concurrency, performance, databases, "
        "distributed systems, backend/frontend systems, "
        "or multi-component implementation.",

    "TRANSFORM":
        "Tasks whose main purpose is transforming existing "
        "user-provided content, including translation, "
        "summarization, rewriting, extraction, formatting, "
        "shortening, restructuring, or conversion.",

    "CREATIVE":
        "Creative generation and ideation such as stories, "
        "dialogue, slogans, names, concepts, fictional content, "
        "brainstorming, characters, scenarios, or creative variations.",
}


# =========================================================
# BOUNDARY MAP
# =========================================================

CONFUSION_BOUNDARIES = {

    "SIMPLE": [
        "GENERAL",
        "TRANSFORM",
        "CREATIVE",
        "REASONING",
    ],

    "GENERAL": [
        "SIMPLE",
        "REASONING",
        "CODING_COMPLEX",
    ],

    "REASONING": [
        "CODING_COMPLEX",
        "GENERAL",
    ],

    "CODING_SIMPLE": [
        "CODING_COMPLEX",
        "REASONING",
    ],

    "CODING_COMPLEX": [
        "REASONING",
        "GENERAL",
        "CODING_SIMPLE",
    ],

    "TRANSFORM": [
        "SIMPLE",
        "CREATIVE",
    ],

    "CREATIVE": [
        "TRANSFORM",
        "SIMPLE",
    ],
}


# =========================================================
# VALIDATION
# =========================================================

for label in LABELS:

    if label not in CATEGORY_DESCRIPTIONS:

        raise ValueError(
            f"Missing category description: {label}"
        )


print()
print("=" * 65)
print("ROUTING TAXONOMY")
print("=" * 65)

for i, label in enumerate(
    LABELS,
    start=1
):

    print(
        f"{i}. {label}"
    )

print()
print(
    "Total labels:",
    len(LABELS)
)

print("=" * 65)


ROUTING TAXONOMY
1. SIMPLE
2. GENERAL
3. REASONING
4. CODING_SIMPLE
5. CODING_COMPLEX
6. TRANSFORM
7. CREATIVE

Total labels: 7


In [4]:
# =========================================================
# CELL 4 — LOAD EMBEDDING MODEL
# ROUTER AUTO TUNE V2
# =========================================================

from sentence_transformers import SentenceTransformer


# =========================================================
# MODEL CONFIG
# =========================================================

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)


# =========================================================
# LOAD MODEL
# =========================================================

print()
print("=" * 65)
print("LOADING EMBEDDING MODEL")
print("=" * 65)

print(
    "Model:",
    EMBEDDING_MODEL_NAME
)


embedding_model = (
    SentenceTransformer(
        EMBEDDING_MODEL_NAME
    )
)


print()
print(
    "Embedding model loaded ✅"
)

print("=" * 65)


# =========================================================
# QUICK TEST
# =========================================================

test_embedding = (
    embedding_model.encode(
        [
            "apa itu HTTP",
            "buat function python sederhana",
        ],
        normalize_embeddings=True
    )
)


print(
    "Embedding shape:",
    test_embedding.shape
)

print(
    "Embedding dimension:",
    test_embedding.shape[1]
)

print(
    "Finite values:",
    bool(
        np.isfinite(
            test_embedding
        ).all()
    )
)


LOADING EMBEDDING MODEL
Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded ✅
Embedding shape: (2, 384)
Embedding dimension: 384
Finite values: True


In [32]:
# Cell 5 — Teacher API + Parser

import requests
import time
import re
import json
import os

# =========================================================
# API CONFIG
# =========================================================

from google.colab import userdata

# BASE_URL = (
#     "https://openrouter.ai/api/v1/chat/completions"
# )

BASE_URL = (
    "https://api.deepseek.com/chat/completions"
)

API_KEY = userdata.get(
    "DEEPSEEK"
)

# API_KEY = userdata.get(
#     "OPEN_ROUTER"
# )

# MODEL = "openrouter/free"
MODEL = "deepseek-v4-flash"

TEMPERATURE = 0
TIMEOUT = 90


if not API_KEY:

    raise RuntimeError(
        "OPEN_ROUTER API key tidak ditemukan "
        "di Colab Secrets."
    )




# =========================================================
# TEACHER SYSTEM PROMPT
# ROUTER AUTO TUNE V2
# =========================================================

SYSTEM_PROMPT = """
You are a strict semantic routing classifier.

Your job is to classify the USER REQUEST into exactly ONE
of the routing labels below.

You are NOT answering the user's request.

You are ONLY classifying it.

=========================================================
ROUTING LABELS
=========================================================

SIMPLE

Use SIMPLE for:
- simple factual questions
- definitions
- basic lookup-style questions
- simple arithmetic
- short tasks requiring almost no reasoning

Examples:
- "apa itu HTTP"
- "berapa 15 persen dari 500"
- "siapa penemu Python"

Do NOT use SIMPLE when:
- substantial explanation is requested
- comparison or planning is required
- actual coding is requested
- existing content must be transformed


GENERAL

Use GENERAL for:
- normal explanations
- ordinary informational questions
- conceptual explanations
- questions requiring understanding but not substantial
  multi-step reasoning

Examples:
- "jelaskan cara kerja DNS secara sederhana"
- "apa fungsi docker compose"
- "kenapa langit terlihat biru"

GENERAL is NOT REASONING.

Use REASONING only when the request clearly requires:
- comparison
- trade-offs
- deduction
- planning
- decision making
- multi-step analysis


REASONING

Use REASONING for NON-CODING analytical tasks requiring:
- comparison
- planning
- trade-offs
- deduction
- mathematics
- decision making
- multi-step reasoning

Examples:
- "lebih baik membeli laptop A atau B untuk kebutuhan saya"
- "bandingkan REST dan GraphQL untuk aplikasi besar"
- "pilih strategi terbaik berdasarkan kelebihan dan kekurangan ini"

Important:

A technology-related question is NOT automatically CODING_COMPLEX.

Generic technology comparison, architecture concepts,
hardware comparison, or technical decision making without
debugging or implementing an actual software system is REASONING.


CODING_SIMPLE

Use CODING_SIMPLE when the user actually requests a small
programming operation such as:
- short function
- regex
- simple SQL
- syntax correction
- short script
- simple code modification
- straightforward implementation

Examples:
- "buat regex validasi email"
- "perbaiki syntax SQL ini"
- "buat function Python untuk parsing JSON"

A factual question about programming is not automatically
CODING_SIMPLE.


CODING_COMPLEX

Use CODING_COMPLEX for substantial SOFTWARE ENGINEERING work.

Examples include:
- debugging complex systems
- backend/frontend architecture
- system design
- concurrency
- race conditions
- security review
- performance optimization
- distributed systems
- databases
- infrastructure
- multi-component implementation
- debugging production systems

Examples:
- "debug kenapa service systemd saya restart terus"
- "review architecture backend saya dan cari bottleneck"
- "analisis race condition pada worker ini"

Important boundary:

REASONING:
generic technical comparison or decision making.

CODING_COMPLEX:
actual software engineering implementation, debugging,
architecture review, optimization, security analysis,
or system-level development.


TRANSFORM

Use TRANSFORM when the user's primary request is to transform
content that they already provide.

Examples:
- translation
- summarization
- rewriting
- extraction
- formatting
- shortening
- restructuring
- converting text/data into another form

Examples:
- "terjemahkan kalimat ini ke Jepang"
- "ringkas artikel berikut"
- "ubah JSON ini menjadi tabel"

Important:
The content to transform may appear after the instruction.


CREATIVE

Use CREATIVE for:
- creative writing
- stories
- dialogue
- names
- slogans
- fictional concepts
- brainstorming
- imaginative scenarios
- creative variations

Examples:
- "buat cerita tentang robot kecil"
- "beri 10 ide nama untuk AI assistant"
- "buat slogan untuk produk saya"


=========================================================
CLASSIFICATION PRIORITY
=========================================================

When multiple labels seem possible, classify based on the
PRIMARY ACTION requested by the user.

Examples:

"jelaskan regex"
-> GENERAL

"buat regex"
-> CODING_SIMPLE


"jelaskan microservices"
-> GENERAL

"bandingkan monolith dan microservices untuk startup saya"
-> REASONING

"desain architecture microservices untuk production backend"
-> CODING_COMPLEX


"apa arti kalimat Jepang ini"
-> TRANSFORM

"apa itu bahasa Jepang"
-> SIMPLE or GENERAL depending on requested explanation depth


"buat nama robot"
-> CREATIVE

"pilih nama terbaik dari daftar ini berdasarkan target market"
-> REASONING


=========================================================
DIFFICULTY
=========================================================

Return difficulty from 1 to 5.

1 = trivial
2 = easy
3 = moderate
4 = difficult
5 = highly complex


=========================================================
CONFIDENCE
=========================================================

Return confidence between 0 and 1.

Use high confidence only when the routing label is clear.

If the request sits close to a category boundary,
use lower confidence.


=========================================================
OUTPUT FORMAT
=========================================================

Return ONLY valid JSON.

Do not use markdown.

Do not explain your reasoning.

Do not answer the user request.

Exact schema:

{
  "label": "LABEL",
  "difficulty": 1,
  "confidence": 0.95
}

Allowed LABEL values:

SIMPLE
GENERAL
REASONING
CODING_SIMPLE
CODING_COMPLEX
TRANSFORM
CREATIVE
"""

# Call Api

def call_teacher(prompt):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

# DEEP
    payload = {
        "model": MODEL,
        "temperature": TEMPERATURE,

        # Untuk task labeling kita tidak butuh reasoning panjang
        "thinking": {
            "type": "disabled"
        },

        # Memaksa output JSON valid
        "response_format": {
            "type": "json_object"
        },

        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
    }

# OPEN ROUTER
    # payload = {
    #     "model": MODEL,
    #     "temperature": 0,
    #     "messages": [
    #         {
    #             "role": "system",
    #             "content": SYSTEM_PROMPT
    #         },
    #         {
    #             "role": "user",
    #             "content": prompt
    #         }
    #     ]
    # }

    print("  -> POST", BASE_URL)
    print("  -> model:", MODEL)

    start = time.time()

    response = requests.post(
        BASE_URL,
        headers=headers,
        json=payload,

        # jangan 90 detik dulu untuk debugging
        timeout=(10, 30)
    )

    print(
        f"  <- HTTP {response.status_code}"
        f" ({time.time() - start:.2f}s)"
    )

    response.raise_for_status()

    data = response.json()

    return data["choices"][0]["message"]["content"]


# Parser JSON yang lebih tahan error



def parse_teacher_output(text):
    if not text:
        raise ValueError("Teacher mengembalikan response kosong")

    text = text.strip()

    # Hapus markdown fence kalau model memberi ```json
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    # Cari object JSON
    match = re.search(r'\{[\s\S]*?\}', text)

    if not match:
        raise ValueError(
            f"Teacher tidak mengembalikan JSON. Raw: {text[:300]!r}"
        )

    try:
        data = json.loads(match.group())
    except json.JSONDecodeError as e:
        raise ValueError(
            f"JSON teacher rusak: {match.group()[:300]}"
        ) from e

    label = data.get("label")

    try:
        difficulty = int(data.get("difficulty"))
        confidence = float(data.get("confidence"))
    except (TypeError, ValueError):
        raise ValueError(
            f"difficulty/confidence invalid: {data}"
        )

    if label not in LABELS:
        raise ValueError(f"Label invalid: {label}")

    if not 1 <= difficulty <= 5:
        raise ValueError(f"Difficulty invalid: {difficulty}")

    if not 0 <= confidence <= 1:
        raise ValueError(f"Confidence invalid: {confidence}")

    return {
        "label": label,
        "difficulty": difficulty,
        "confidence": confidence,
    }


# =========================================================
# CELL 5 — TEACHER SANITY CHECK
# ROUTER AUTO TUNE V2
# =========================================================

required_teacher_functions = [
    "call_teacher",
    "parse_teacher_output",
]


missing = [
    name
    for name
    in required_teacher_functions
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Teacher setup belum lengkap: "
        + ", ".join(
            missing
        )
    )


print()
print("=" * 65)
print("TEACHER SETUP")
print("=" * 65)

print(
    "call_teacher         : OK"
)

print(
    "parse_teacher_output : OK"
)

print("=" * 65)




TEACHER SETUP
call_teacher         : OK
parse_teacher_output : OK


In [6]:
# # =========================================================
# # QUICK TEACHER TEST
# # =========================================================

# print()
# print("Testing teacher API...")

# test_raw = call_teacher(
#     "berapa 10 persen dari 200"
# )

# print()
# print("RAW:")
# print(test_raw)

# test_parsed = parse_teacher_output(
#     test_raw
# )

# print()
# print("PARSED:")
# print(test_parsed)

# print()
# print("Teacher API test OK ✅")

In [7]:
# =========================================================
# CELL 6 — LOAD V1 BASELINE DATASET
# ROUTER AUTO TUNE V2
# =========================================================


def load_v1_baseline():

    # -----------------------------------------------------
    # PRIORITY 1:
    # gunakan dataset hasil gabungan terakhir dari V1
    # -----------------------------------------------------

    if os.path.exists(
        CURRENT_DATASET_PATH
    ):

        print(
            "Loading V1 current dataset:"
        )

        print(
            CURRENT_DATASET_PATH
        )

        df = pd.read_csv(
            CURRENT_DATASET_PATH
        )

    # -----------------------------------------------------
    # FALLBACK:
    # merge base + extra jika current dataset tidak ada
    # -----------------------------------------------------

    else:

        print(
            "CURRENT_DATASET_PATH tidak ditemukan."
        )

        print(
            "Fallback merge BASE + EXTRA."
        )

        if not os.path.exists(
            BASE_PATH
        ):

            raise FileNotFoundError(
                "Baseline dataset tidak ditemukan:\n"
                + BASE_PATH
            )

        df_base = pd.read_csv(
            BASE_PATH
        )

        if os.path.exists(
            EXTRA_ALL_PATH
        ):

            df_extra = pd.read_csv(
                EXTRA_ALL_PATH
            )

            df = pd.concat(
                [
                    df_base,
                    df_extra,
                ],
                ignore_index=True
            )

        else:

            df = df_base.copy()


    # =====================================================
    # REQUIRED COLUMNS
    # =====================================================

    required = {
        "prompt",
        "label",
    }

    missing_columns = (
        required
        - set(
            df.columns
        )
    )

    if missing_columns:

        raise ValueError(
            "Dataset kurang kolom: "
            + ", ".join(
                sorted(
                    missing_columns
                )
            )
        )


    # =====================================================
    # CLEAN
    # =====================================================

    df["prompt"] = (
        df["prompt"]
        .astype(str)
        .str.strip()
    )

    df["label"] = (
        df["label"]
        .astype(str)
        .str.strip()
    )

    df = df[
        df["prompt"] != ""
    ]

    df = df[
        df["label"].isin(
            LABELS
        )
    ]


    # -----------------------------------------------------
    # Filter status kalau tersedia
    # -----------------------------------------------------

    if "status" in df.columns:

        df = df[
            df["status"] == "ok"
        ]


    # -----------------------------------------------------
    # Filter confidence kalau tersedia
    # -----------------------------------------------------

    if "confidence" in df.columns:

        df["confidence"] = (
            pd.to_numeric(
                df["confidence"],
                errors="coerce"
            )
        )

        df = df[
            (
                df["confidence"].isna()
            )
            |
            (
                df["confidence"]
                >= MIN_CONFIDENCE
            )
        ]


    # =====================================================
    # REMOVE DUPLICATES
    # =====================================================

    df = (
        df
        .drop_duplicates(
            subset=["prompt"],
            keep="first"
        )
        .reset_index(
            drop=True
        )
    )


    return df


# =========================================================
# LOAD BASELINE
# =========================================================

df_v1_baseline = (
    load_v1_baseline()
)


print()
print("=" * 65)
print("V1 BASELINE DATASET")
print("=" * 65)

print(
    "Samples:",
    len(
        df_v1_baseline
    )
)

print()

print(
    df_v1_baseline[
        "label"
    ]
    .value_counts()
    .reindex(
        LABELS,
        fill_value=0
    )
)

print("=" * 65)

Loading V1 current dataset:
/content/drive/MyDrive/router_classifier/router_dataset_current.csv

V1 BASELINE DATASET
Samples: 473

label
SIMPLE             80
GENERAL            81
REASONING         114
CODING_SIMPLE      52
CODING_COMPLEX     56
TRANSFORM          50
CREATIVE           40
Name: count, dtype: int64


In [8]:
# =========================================================
# CELL 7 — BUILD / LOAD FIXED VALIDATION SET
# ROUTER AUTO TUNE V2
# =========================================================

from sklearn.model_selection import train_test_split


VALIDATION_SIZE = 0.20


def build_or_load_v2_validation(
    df
):

    # -----------------------------------------------------
    # LOAD EXISTING FIXED VALIDATION
    # -----------------------------------------------------

    if os.path.exists(
        V2_VALIDATION_PATH
    ):

        print(
            "Loading existing V2 fixed validation:"
        )

        print(
            V2_VALIDATION_PATH
        )

        df_val = pd.read_csv(
            V2_VALIDATION_PATH
        )


        required = {
            "prompt",
            "label",
        }


        missing = (
            required
            - set(
                df_val.columns
            )
        )


        if missing:

            raise ValueError(
                "Validation file invalid. Missing: "
                + ", ".join(
                    sorted(
                        missing
                    )
                )
            )


        df_val["prompt"] = (
            df_val["prompt"]
            .astype(str)
            .str.strip()
        )


        df_val["label"] = (
            df_val["label"]
            .astype(str)
            .str.strip()
        )


        df_val = (
            df_val[
                df_val[
                    "label"
                ].isin(
                    LABELS
                )
            ]
            .drop_duplicates(
                subset=["prompt"],
                keep="first"
            )
            .reset_index(
                drop=True
            )
        )


        print()
        print(
            "Existing fixed validation loaded ✅"
        )

        return df_val


    # -----------------------------------------------------
    # CREATE NEW FIXED VALIDATION
    # -----------------------------------------------------

    print(
        "Creating NEW V2 fixed validation..."
    )


    counts = (
        df[
            "label"
        ]
        .value_counts()
    )


    print()
    print(
        "Baseline class counts:"
    )

    print(
        counts.reindex(
            LABELS,
            fill_value=0
        )
    )


    if counts.min() < 2:

        raise RuntimeError(
            "Ada class dengan < 2 sample. "
            "Tidak bisa membuat stratified validation."
        )


    _, df_val = train_test_split(
        df,
        test_size=VALIDATION_SIZE,
        random_state=RANDOM_STATE,
        stratify=df["label"]
    )


    df_val = (
        df_val[
            [
                "prompt",
                "label",
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    df_val.to_csv(
        V2_VALIDATION_PATH,
        index=False
    )


    print()
    print(
        "Fixed validation created ✅"
    )

    print(
        "Saved:",
        V2_VALIDATION_PATH
    )


    return df_val


# =========================================================
# BUILD / LOAD
# =========================================================

df_v2_validation = (
    build_or_load_v2_validation(
        df_v1_baseline
    )
)


# =========================================================
# DISPLAY STATUS
# =========================================================

print()
print("=" * 65)
print("V2 FIXED VALIDATION")
print("=" * 65)

print(
    "Validation samples:",
    len(
        df_v2_validation
    )
)

print()

print(
    df_v2_validation[
        "label"
    ]
    .value_counts()
    .reindex(
        LABELS,
        fill_value=0
    )
)

print("=" * 65)

Loading existing V2 fixed validation:
/content/drive/MyDrive/router_classifier/router_v2_validation_fixed.csv

Existing fixed validation loaded ✅

V2 FIXED VALIDATION
Validation samples: 95

label
SIMPLE            16
GENERAL           16
REASONING         23
CODING_SIMPLE     11
CODING_COMPLEX    11
TRANSFORM         10
CREATIVE           8
Name: count, dtype: int64


In [9]:
# =========================================================
# CELL 8 — BUILD BASELINE TRAINING SET
# ROUTER AUTO TUNE V2
# =========================================================


def remove_validation_samples(
    df,
    df_validation
):

    validation_prompts = set(
        df_validation[
            "prompt"
        ]
        .astype(str)
        .str.strip()
        .str.casefold()
    )


    df_train = (
        df[
            ~df[
                "prompt"
            ]
            .astype(str)
            .str.strip()
            .str.casefold()
            .isin(
                validation_prompts
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    return df_train


df_v2_train_baseline = (
    remove_validation_samples(
        df_v1_baseline,
        df_v2_validation
    )
)


print()
print("=" * 65)
print("V2 BASELINE SPLIT")
print("=" * 65)

print(
    "Full baseline :",
    len(
        df_v1_baseline
    )
)

print(
    "Train         :",
    len(
        df_v2_train_baseline
    )
)

print(
    "Validation    :",
    len(
        df_v2_validation
    )
)

print()

print(
    "Overlap check :",
    len(
        set(
            df_v2_train_baseline["prompt"]
        )
        &
        set(
            df_v2_validation["prompt"]
        )
    )
)

print("=" * 65)


V2 BASELINE SPLIT
Full baseline : 473
Train         : 378
Validation    : 95

Overlap check : 0


In [10]:
# =========================================================
# CELL 9 — TRAIN BASELINE MODEL
# ROUTER AUTO TUNE V2
# =========================================================

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


def train_and_evaluate_router(
    df_train,
    df_validation
):

    if len(df_train) == 0:

        raise RuntimeError(
            "Training dataset kosong."
        )

    if len(df_validation) == 0:

        raise RuntimeError(
            "Validation dataset kosong."
        )


    # =====================================================
    # EMBEDDING TRAIN
    # =====================================================

    print()
    print("Generating TRAIN embeddings...")

    X_train = (
        embedding_model.encode(
            df_train[
                "prompt"
            ].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True
        )
    )

    y_train = (
        df_train[
            "label"
        ]
        .to_numpy()
    )


    # =====================================================
    # EMBEDDING VALIDATION
    # =====================================================

    print()
    print("Generating VALIDATION embeddings...")

    X_val = (
        embedding_model.encode(
            df_validation[
                "prompt"
            ].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True
        )
    )

    y_val = (
        df_validation[
            "label"
        ]
        .to_numpy()
    )


    # =====================================================
    # CHECK CLASS COUNTS
    # =====================================================

    counts = (
        pd.Series(
            y_train
        )
        .value_counts()
    )

    print()
    print("Training distribution:")

    print(
        counts.reindex(
            LABELS,
            fill_value=0
        )
    )


    if counts.min() < 2:

        raise RuntimeError(
            "Ada class training dengan < 2 sample."
        )


    # =====================================================
    # GRID SEARCH C
    # =====================================================

    cv_splits = min(
        5,
        int(
            counts.min()
        )
    )

    cv = StratifiedKFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )


    base_model = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )


    search = GridSearchCV(
        estimator=base_model,

        param_grid={
            "C": GRID_C_VALUES
        },

        scoring="f1_macro",

        cv=cv,

        n_jobs=-1,

        verbose=0
    )


    print()
    print("Searching best C...")

    search.fit(
        X_train,
        y_train
    )


    classifier = (
        search.best_estimator_
    )


    # =====================================================
    # VALIDATION PREDICTION
    # =====================================================

    y_pred = (
        classifier.predict(
            X_val
        )
    )


    accuracy = (
        accuracy_score(
            y_val,
            y_pred
        )
    )


    macro_f1 = (
        f1_score(
            y_val,
            y_pred,
            average="macro",
            zero_division=0
        )
    )


    report = classification_report(
        y_val,
        y_pred,
        labels=LABELS,
        output_dict=True,
        zero_division=0
    )


    cm = confusion_matrix(
        y_val,
        y_pred,
        labels=LABELS
    )


    per_class_recall = {}

    per_class_f1 = {}


    for label in LABELS:

        class_result = (
            report.get(
                label,
                {}
            )
        )


        per_class_recall[
            label
        ] = float(
            class_result.get(
                "recall",
                0
            )
        )


        per_class_f1[
            label
        ] = float(
            class_result.get(
                "f1-score",
                0
            )
        )


    metrics = {

        "accuracy":
            float(
                accuracy
            ),

        "macro_f1":
            float(
                macro_f1
            ),

        "cv_macro_f1":
            float(
                search.best_score_
            ),

        "best_c":
            float(
                search.best_params_[
                    "C"
                ]
            ),

        "validation_samples":
            int(
                len(
                    y_val
                )
            ),

        "per_class_recall":
            per_class_recall,

        "per_class_f1":
            per_class_f1,
    }


    evaluation = {

        "y_true":
            y_val,

        "y_pred":
            y_pred,

        "confusion_matrix":
            cm,

        "report":
            report,
    }


    return (
        classifier,
        metrics,
        evaluation
    )


# =========================================================
# TRAIN BASELINE
# =========================================================

print()
print("=" * 65)
print("TRAINING V2 BASELINE")
print("=" * 65)


(
    baseline_classifier,
    baseline_metrics,
    baseline_evaluation,
) = train_and_evaluate_router(
    df_v2_train_baseline,
    df_v2_validation
)


# =========================================================
# RESULT
# =========================================================

print()
print("=" * 65)
print("V2 BASELINE RESULT")
print("=" * 65)

print(
    "Accuracy :",
    round(
        baseline_metrics[
            "accuracy"
        ],
        4
    )
)

print(
    "Macro F1 :",
    round(
        baseline_metrics[
            "macro_f1"
        ],
        4
    )
)

print(
    "CV F1    :",
    round(
        baseline_metrics[
            "cv_macro_f1"
        ],
        4
    )
)

print(
    "Best C   :",
    baseline_metrics[
        "best_c"
    ]
)

print()

print("Per-class recall:")

for label in LABELS:

    print(
        f"{label:16} "
        f"{baseline_metrics['per_class_recall'][label]:.3f}"
    )

print("=" * 65)


TRAINING V2 BASELINE

Generating TRAIN embeddings...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


Generating VALIDATION embeddings...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Training distribution:
SIMPLE            64
GENERAL           65
REASONING         91
CODING_SIMPLE     41
CODING_COMPLEX    45
TRANSFORM         40
CREATIVE          32
Name: count, dtype: int64

Searching best C...

V2 BASELINE RESULT
Accuracy : 0.6947
Macro F1 : 0.7036
CV F1    : 0.634
Best C   : 3.0

Per-class recall:
SIMPLE           0.750
GENERAL          0.438
REASONING        0.565
CODING_SIMPLE    0.909
CODING_COMPLEX   0.909
TRANSFORM        0.800
CREATIVE         0.750


In [11]:
# =========================================================
# CELL 10 — SAVE V2 BASELINE
# =========================================================

import joblib
import json


joblib.dump(
    baseline_classifier,
    V2_CURRENT_MODEL_PATH
)


joblib.dump(
    baseline_classifier,
    V2_BEST_MODEL_PATH
)


with open(
    V2_BEST_METRICS_PATH,
    "w"
) as f:

    json.dump(
        baseline_metrics,
        f,
        indent=2
    )


df_v2_train_baseline.to_csv(
    V2_ACTIVE_DATASET_PATH,
    index=False
)


print()
print("=" * 65)
print("V2 BASELINE SAVED")
print("=" * 65)

print(
    "Active dataset :",
    V2_ACTIVE_DATASET_PATH
)

print(
    "Current model  :",
    V2_CURRENT_MODEL_PATH
)

print(
    "Best model     :",
    V2_BEST_MODEL_PATH
)

print(
    "Best metrics   :",
    V2_BEST_METRICS_PATH
)

print("=" * 65)


V2 BASELINE SAVED
Active dataset : /content/drive/MyDrive/router_classifier/router_v2_active_dataset.csv
Current model  : /content/drive/MyDrive/router_classifier/router_v2_current.joblib
Best model     : /content/drive/MyDrive/router_classifier/router_v2_best.joblib
Best metrics   : /content/drive/MyDrive/router_classifier/router_v2_best_metrics.json


In [12]:
# =========================================================
# CELL 11 — DETECT BIGGEST CONFUSION
# ROUTER AUTO TUNE V2
# =========================================================

def detect_biggest_confusion(
    evaluation,
    metrics
):

    cm = np.asarray(
        evaluation[
            "confusion_matrix"
        ]
    )

    recalls = (
        metrics[
            "per_class_recall"
        ]
    )

    candidates = []

    for i, true_label in enumerate(
        LABELS
    ):

        row = cm[i]

        wrong = []

        for j, count in enumerate(
            row
        ):

            if i == j:
                continue

            if count <= 0:
                continue

            wrong.append(
                {
                    "predicted_label": LABELS[j],
                    "count": int(count),
                }
            )

        wrong = sorted(
            wrong,
            key=lambda x: x["count"],
            reverse=True
        )

        top_confusion = (
            wrong[0]
            if wrong
            else None
        )

        candidates.append(
            {
                "true_label": true_label,
                "recall": float(
                    recalls.get(
                        true_label,
                        0
                    )
                ),
                "top_confusion": top_confusion,
                "total_wrong": int(
                    row.sum()
                    - row[i]
                ),
            }
        )

    # Prioritas:
    # recall rendah + jumlah salah besar
    candidates = sorted(
        candidates,
        key=lambda x: (
            x["recall"],
            -x["total_wrong"]
        )
    )

    return candidates


confusion_ranking = (
    detect_biggest_confusion(
        baseline_evaluation,
        baseline_metrics
    )
)


print()
print("=" * 65)
print("CONFUSION RANKING")
print("=" * 65)

for item in confusion_ranking:

    print()

    print(
        "TRUE LABEL :",
        item[
            "true_label"
        ]
    )

    print(
        "Recall     :",
        round(
            item[
                "recall"
            ],
            3
        )
    )

    print(
        "Total wrong:",
        item[
            "total_wrong"
        ]
    )

    print(
        "Top confusion:",
        item[
            "top_confusion"
        ]
    )

print("=" * 65)


CONFUSION RANKING

TRUE LABEL : GENERAL
Recall     : 0.438
Total wrong: 9
Top confusion: {'predicted_label': 'SIMPLE', 'count': 3}

TRUE LABEL : REASONING
Recall     : 0.565
Total wrong: 10
Top confusion: {'predicted_label': 'CODING_COMPLEX', 'count': 7}

TRUE LABEL : SIMPLE
Recall     : 0.75
Total wrong: 4
Top confusion: {'predicted_label': 'CREATIVE', 'count': 2}

TRUE LABEL : CREATIVE
Recall     : 0.75
Total wrong: 2
Top confusion: {'predicted_label': 'GENERAL', 'count': 1}

TRUE LABEL : TRANSFORM
Recall     : 0.8
Total wrong: 2
Top confusion: {'predicted_label': 'CODING_SIMPLE', 'count': 1}

TRUE LABEL : CODING_SIMPLE
Recall     : 0.909
Total wrong: 1
Top confusion: {'predicted_label': 'REASONING', 'count': 1}

TRUE LABEL : CODING_COMPLEX
Recall     : 0.909
Total wrong: 1
Top confusion: {'predicted_label': 'REASONING', 'count': 1}


In [13]:
# =========================================================
# CELL 12 — SELECT NEXT TUNING TARGET
# ROUTER AUTO TUNE V2
# =========================================================


def select_tuning_target(
    confusion_ranking
):

    for item in confusion_ranking:

        if (
            item[
                "top_confusion"
            ]
            is None
        ):
            continue

        if (
            item[
                "total_wrong"
            ]
            <= 0
        ):
            continue

        return {
            "target_label":
                item[
                    "true_label"
                ],

            "confusing_with":
                item[
                    "top_confusion"
                ][
                    "predicted_label"
                ],

            "wrong_count":
                item[
                    "top_confusion"
                ][
                    "count"
                ],

            "current_recall":
                item[
                    "recall"
                ],
        }

    return None


tuning_target = (
    select_tuning_target(
        confusion_ranking
    )
)


print()
print("=" * 65)
print("NEXT TUNING TARGET")
print("=" * 65)

if tuning_target is None:

    print(
        "Tidak ada confusion target."
    )

else:

    print(
        "Target label   :",
        tuning_target[
            "target_label"
        ]
    )

    print(
        "Confusing with :",
        tuning_target[
            "confusing_with"
        ]
    )

    print(
        "Wrong samples  :",
        tuning_target[
            "wrong_count"
        ]
    )

    print(
        "Current recall :",
        round(
            tuning_target[
                "current_recall"
            ],
            3
        )
    )

print("=" * 65)


NEXT TUNING TARGET
Target label   : GENERAL
Confusing with : SIMPLE
Wrong samples  : 3
Current recall : 0.438


In [15]:
# =========================================================
# CELL 13 — GENERATE CANDIDATE BATCH
# ROUTER AUTO TUNE V2
# =========================================================


def generate_candidate_prompts(
    target_label,
    confusing_with,
    count=CANDIDATE_BATCH_SIZE
):

    target_definition = (
        CATEGORY_DESCRIPTIONS[
            target_label
        ]
    )

    confusing_definition = (
        CATEGORY_DESCRIPTIONS[
            confusing_with
        ]
    )

    request = f"""
Generate exactly {count} realistic USER prompts for a
semantic routing classifier.

TARGET LABEL:
{target_label}

TARGET DEFINITION:
{target_definition}

CONFUSING NEIGHBOR:
{confusing_with}

NEIGHBOR DEFINITION:
{confusing_definition}

The goal is to create HARD BOUNDARY examples.

Each prompt should look somewhat similar to {confusing_with},
but must genuinely belong to {target_label}.

Important:
- correct label MUST be {target_label}
- do not generate prompts that actually belong to {confusing_with}
- use realistic user language
- vary topic and sentence structure
- approximately 60% Indonesian
- approximately 25% English
- approximately 15% mixed Indonesian-English
- no duplicates
- do not answer the prompts
- do not mention the routing label
- return ONLY a valid JSON array of strings

Example output:

[
  "prompt 1",
  "prompt 2"
]
"""

    raw = call_teacher(
        request
    )

    if not raw:

        raise RuntimeError(
            "Teacher response kosong."
        )

    cleaned = (
        str(raw)
        .strip()
    )

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.I
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    match = re.search(
        r"\[[\s\S]*\]",
        cleaned
    )

    if not match:

        raise ValueError(
            "JSON array tidak ditemukan:\n"
            + cleaned[:500]
        )

    data = json.loads(
        match.group()
    )

    if not isinstance(
        data,
        list
    ):

        raise ValueError(
            "Teacher output bukan list."
        )

    clean = []

    seen = set()

    for item in data:

        if not isinstance(
            item,
            str
        ):
            continue

        item = item.strip()

        if not item:
            continue

        normalized = (
            item.casefold()
        )

        if normalized in seen:
            continue

        seen.add(
            normalized
        )

        clean.append(
            item
        )

    return clean


if tuning_target is None:

    candidate_prompts = []

else:

    candidate_prompts = (
        generate_candidate_prompts(
            target_label=tuning_target[
                "target_label"
            ],
            confusing_with=tuning_target[
                "confusing_with"
            ],
            count=CANDIDATE_BATCH_SIZE
        )
    )


print()
print("=" * 65)
print("GENERATED CANDIDATES")
print("=" * 65)

for i, prompt in enumerate(
    candidate_prompts,
    start=1
):

    print(
        f"{i}. {prompt}"
    )

print()
print(
    "Generated:",
    len(
        candidate_prompts
    )
)

print("=" * 65)

  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (41.95s)

GENERATED CANDIDATES
1. jelaskan mengapa HTTPS lebih aman dibanding HTTP
2. bagaimana cara kerja load balancer secara singkat
3. apa yang terjadi saat browser memproses HTML hingga tampil di layar
4. explain how DNS resolution works in simple terms
5. what is the purpose of a reverse proxy in web architecture
6. jelaskan kenapa database index mempercepat query

Generated: 6


In [16]:
# =========================================================
# CELL 14 — VALIDATE CANDIDATE BATCH
# ROUTER AUTO TUNE V2
# =========================================================


def validate_candidate_batch(
    prompts,
    expected_label
):

    results = []

    for prompt in tqdm(
        prompts,
        desc=f"Validate {expected_label}"
    ):

        success = False
        last_error = None

        for attempt in range(3):

            try:

                raw = call_teacher(
                    prompt
                )

                parsed = (
                    parse_teacher_output(
                        raw
                    )
                )

                predicted_label = (
                    parsed["label"]
                )

                confidence = float(
                    parsed["confidence"]
                )

                difficulty = int(
                    parsed["difficulty"]
                )


                # -----------------------------------------
                # REJECT LABEL MISMATCH
                # -----------------------------------------

                if (
                    predicted_label
                    != expected_label
                ):

                    results.append({
                        "prompt":
                            prompt,

                        "label":
                            predicted_label,

                        "expected_label":
                            expected_label,

                        "difficulty":
                            difficulty,

                        "confidence":
                            confidence,

                        "status":
                            "label_mismatch",
                    })

                    success = True
                    break


                # -----------------------------------------
                # REJECT LOW CONFIDENCE
                # -----------------------------------------

                if (
                    confidence
                    < MIN_CONFIDENCE
                ):

                    results.append({
                        "prompt":
                            prompt,

                        "label":
                            predicted_label,

                        "expected_label":
                            expected_label,

                        "difficulty":
                            difficulty,

                        "confidence":
                            confidence,

                        "status":
                            "low_confidence",
                    })

                    success = True
                    break


                # -----------------------------------------
                # ACCEPT
                # -----------------------------------------

                results.append({
                    "prompt":
                        prompt,

                    "label":
                        predicted_label,

                    "expected_label":
                        expected_label,

                    "difficulty":
                        difficulty,

                    "confidence":
                        confidence,

                    "status":
                        "ok",
                })

                success = True
                break


            except Exception as e:

                last_error = e

                print(
                    "\nValidation error:",
                    type(e).__name__,
                    str(e)[:300]
                )

                if attempt < 2:

                    time.sleep(
                        2 * (attempt + 1)
                    )


        if not success:

            results.append({
                "prompt":
                    prompt,

                "label":
                    None,

                "expected_label":
                    expected_label,

                "difficulty":
                    None,

                "confidence":
                    None,

                "status":
                    "error",
            })

            print(
                "Validation failed permanently:",
                prompt[:100],
                last_error
            )


    return results


# =========================================================
# RUN VALIDATION
# =========================================================

if tuning_target is None:

    candidate_validation_results = []

else:

    candidate_validation_results = (
        validate_candidate_batch(
            prompts=candidate_prompts,
            expected_label=tuning_target[
                "target_label"
            ]
        )
    )


# =========================================================
# SPLIT ACCEPT / REJECT
# =========================================================

accepted_candidates = [
    item
    for item
    in candidate_validation_results
    if item[
        "status"
    ] == "ok"
]


rejected_candidates = [
    item
    for item
    in candidate_validation_results
    if item[
        "status"
    ] != "ok"
]


print()
print("=" * 65)
print("CANDIDATE VALIDATION RESULT")
print("=" * 65)

print(
    "Generated :",
    len(
        candidate_prompts
    )
)

print(
    "Accepted  :",
    len(
        accepted_candidates
    )
)

print(
    "Rejected  :",
    len(
        rejected_candidates
    )
)

print()


for item in accepted_candidates:

    print(
        "[ACCEPT]",
        item["label"],
        "|",
        round(
            float(
                item["confidence"]
            ),
            3
        ),
        "|",
        item["prompt"]
    )


for item in rejected_candidates:

    print(
        "[REJECT]",
        item["status"],
        "|",
        item["prompt"]
    )


print("=" * 65)

Validate GENERAL:   0%|          | 0/6 [00:00<?, ?it/s]

  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (1.77s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (29.84s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (2.11s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (1.10s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (10.63s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (3.72s)

CANDIDATE VALIDATION RESULT
Generated : 6
Accepted  : 5
Rejected  : 1

[ACCEPT] GENERAL | 0.95 | jelaskan mengapa HTTPS lebih aman dibanding HTTP
[ACCEPT] GENERAL | 0.95 | bagaimana cara kerja load balancer secara singkat
[ACCEPT] GENERAL | 0.85 | apa yang terjadi saat browser memproses HTML hingga tampil di layar
[ACCEPT] GENERAL | 0.95 | explain how DNS res

In [17]:
# =========================================================
# CELL 15 — BUILD TEMPORARY CANDIDATE DATASET
# ROUTER AUTO TUNE V2
# =========================================================


def build_candidate_dataframe(
    accepted_results
):

    if not accepted_results:

        return pd.DataFrame(
            columns=[
                "prompt",
                "label",
                "confidence",
                "difficulty",
                "status",
            ]
        )


    rows = []


    for item in accepted_results:

        rows.append({
            "prompt":
                item["prompt"],

            "label":
                item["label"],

            "confidence":
                float(
                    item["confidence"]
                ),

            "difficulty":
                int(
                    item["difficulty"]
                ),

            "status":
                "ok",
        })


    return pd.DataFrame(
        rows
    )


df_candidate_batch = (
    build_candidate_dataframe(
        accepted_candidates
    )
)


print()
print("=" * 65)
print("TEMPORARY CANDIDATE BATCH")
print("=" * 65)

print(
    "Candidate samples:",
    len(
        df_candidate_batch
    )
)

if len(
    df_candidate_batch
) > 0:

    display(
        df_candidate_batch
    )

print("=" * 65)


TEMPORARY CANDIDATE BATCH
Candidate samples: 5


,prompt,label,confidence,difficulty,status
0,jelaskan mengapa HTTPS lebih aman dibanding HTTP,GENERAL,0.95,2,ok
1,bagaimana cara kerja load balancer secara singkat,GENERAL,0.95,2,ok
2,apa yang terjadi saat browser memproses HTML h...,GENERAL,0.85,3,ok
3,explain how DNS resolution works in simple terms,GENERAL,0.95,2,ok
4,jelaskan kenapa database index mempercepat query,GENERAL,0.95,3,ok


In [18]:
# =========================================================
# CELL 16 — BUILD TEMPORARY TRAINING DATASET
# ROUTER AUTO TUNE V2
# =========================================================


if os.path.exists(
    V2_ACTIVE_DATASET_PATH
):

    df_active = pd.read_csv(
        V2_ACTIVE_DATASET_PATH
    )

else:

    df_active = (
        df_v2_train_baseline
        .copy()
    )


print(
    "Active samples before candidate:",
    len(
        df_active
    )
)


if len(
    df_candidate_batch
) > 0:

    df_candidate_train = pd.concat(
        [
            df_active,
            df_candidate_batch,
        ],
        ignore_index=True
    )

else:

    df_candidate_train = (
        df_active.copy()
    )


df_candidate_train = (
    df_candidate_train
    .drop_duplicates(
        subset=["prompt"],
        keep="first"
    )
    .reset_index(
        drop=True
    )
)


print(
    "Temporary candidate dataset:",
    len(
        df_candidate_train
    )
)

print(
    "Added unique samples:",
    len(
        df_candidate_train
    )
    -
    len(
        df_active
    )
)

Active samples before candidate: 378
Temporary candidate dataset: 383
Added unique samples: 5


In [19]:
# =========================================================
# CELL 17 — TRAIN CANDIDATE + QUALITY GATE
# ROUTER AUTO TUNE V2
# =========================================================


if len(
    df_candidate_batch
) == 0:

    candidate_classifier = None
    candidate_metrics = None
    candidate_evaluation = None

    print(
        "Tidak ada accepted candidate."
    )

else:

    print()
    print("=" * 65)
    print("TRAINING CANDIDATE MODEL")
    print("=" * 65)


    (
        candidate_classifier,
        candidate_metrics,
        candidate_evaluation,
    ) = train_and_evaluate_router(
        df_candidate_train,
        df_v2_validation
    )


# =========================================================
# LOAD CURRENT BEST METRICS
# =========================================================

with open(
    V2_BEST_METRICS_PATH,
    "r"
) as f:

    best_metrics = json.load(
        f
    )


best_macro_f1 = float(
    best_metrics[
        "macro_f1"
    ]
)


if candidate_metrics is not None:

    candidate_macro_f1 = float(
        candidate_metrics[
            "macro_f1"
        ]
    )

else:

    candidate_macro_f1 = -1.0


print()
print("=" * 65)
print("QUALITY GATE")
print("=" * 65)

print(
    "Best Macro F1      :",
    round(
        best_macro_f1,
        4
    )
)

print(
    "Candidate Macro F1 :",
    round(
        candidate_macro_f1,
        4
    )
)

print(
    "Required increase  :",
    MIN_IMPROVEMENT
)


# =========================================================
# CHECK TARGET RECALL
# =========================================================

if (
    candidate_metrics is not None
    and
    tuning_target is not None
):

    target_label = (
        tuning_target[
            "target_label"
        ]
    )


    old_target_recall = float(
        best_metrics[
            "per_class_recall"
        ].get(
            target_label,
            0
        )
    )


    new_target_recall = float(
        candidate_metrics[
            "per_class_recall"
        ].get(
            target_label,
            0
        )
    )


else:

    target_label = None
    old_target_recall = 0.0
    new_target_recall = 0.0


print()

print(
    "Target class:",
    target_label
)

print(
    "Old recall  :",
    round(
        old_target_recall,
        4
    )
)

print(
    "New recall  :",
    round(
        new_target_recall,
        4
    )
)


TRAINING CANDIDATE MODEL

Generating TRAIN embeddings...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


Generating VALIDATION embeddings...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Training distribution:
SIMPLE            64
GENERAL           70
REASONING         91
CODING_SIMPLE     41
CODING_COMPLEX    45
TRANSFORM         40
CREATIVE          32
Name: count, dtype: int64

Searching best C...

QUALITY GATE
Best Macro F1      : 0.7036
Candidate Macro F1 : 0.7067
Required increase  : 0.005

Target class: GENERAL
Old recall  : 0.4375
New recall  : 0.4375


In [20]:
# =========================================================
# CELL 18 — CLASS REGRESSION CHECK
# ROUTER AUTO TUNE V2
# =========================================================


class_regressions = []


if candidate_metrics is not None:

    old_recalls = (
        best_metrics[
            "per_class_recall"
        ]
    )

    new_recalls = (
        candidate_metrics[
            "per_class_recall"
        ]
    )


    for label in LABELS:

        old_value = float(
            old_recalls.get(
                label,
                0
            )
        )

        new_value = float(
            new_recalls.get(
                label,
                0
            )
        )

        drop = (
            old_value
            - new_value
        )


        if (
            drop
            > MAX_CLASS_RECALL_DROP
        ):

            class_regressions.append({
                "label":
                    label,

                "old":
                    old_value,

                "new":
                    new_value,

                "drop":
                    drop,
            })


print()
print("=" * 65)
print("CLASS REGRESSION CHECK")
print("=" * 65)


if class_regressions:

    for item in class_regressions:

        print(
            item["label"],
            "|",
            f"{item['old']:.3f}",
            "->",
            f"{item['new']:.3f}",
            "| drop:",
            f"{item['drop']:.3f}"
        )

else:

    print(
        "No major class regression ✅"
    )


print("=" * 65)


CLASS REGRESSION CHECK
CODING_COMPLEX | 0.909 -> 0.818 | drop: 0.091


In [21]:
# =========================================================
# CELL 19 — ACCEPT / REJECT CANDIDATE
# ROUTER AUTO TUNE V2
# =========================================================


accepted_batch = False

decision_reason = ""


if candidate_metrics is None:

    decision_reason = (
        "candidate_training_failed"
    )


else:

    f1_improvement = (
        candidate_macro_f1
        - best_macro_f1
    )


    # -----------------------------------------
    # RULE 1:
    # Macro F1 harus naik cukup
    # -----------------------------------------

    f1_pass = (
        f1_improvement
        >= MIN_IMPROVEMENT
    )


    # -----------------------------------------
    # RULE 2:
    # target recall tidak boleh turun
    # -----------------------------------------

    target_recall_pass = (
        new_target_recall
        >= old_target_recall
    )


    # -----------------------------------------
    # RULE 3:
    # tidak boleh merusak class lain terlalu besar
    # -----------------------------------------

    regression_pass = (
        len(
            class_regressions
        )
        == 0
    )


    accepted_batch = (
        f1_pass
        and
        target_recall_pass
        and
        regression_pass
    )


    if not f1_pass:

        decision_reason += (
            "macro_f1_not_improved;"
        )


    if not target_recall_pass:

        decision_reason += (
            "target_recall_dropped;"
        )


    if not regression_pass:

        decision_reason += (
            "class_regression;"
        )


# =========================================================
# ACCEPT
# =========================================================

if accepted_batch:

    print()
    print("=" * 65)
    print("CANDIDATE ACCEPTED ✅")
    print("=" * 65)


    # Save active dataset
    df_candidate_train.to_csv(
        V2_ACTIVE_DATASET_PATH,
        index=False
    )


    # Append accepted records
    df_accept = (
        df_candidate_batch
        .copy()
    )


    df_accept[
        "accepted_at"
    ] = pd.Timestamp.now()


    df_accept[
        "target_boundary"
    ] = (
        tuning_target[
            "confusing_with"
        ]
    )


    if os.path.exists(
        V2_ACCEPTED_PATH
    ):

        df_old_accept = pd.read_csv(
            V2_ACCEPTED_PATH
        )


        df_accept_all = pd.concat(
            [
                df_old_accept,
                df_accept,
            ],
            ignore_index=True
        )

    else:

        df_accept_all = (
            df_accept
        )


    df_accept_all = (
        df_accept_all
        .drop_duplicates(
            subset=["prompt"],
            keep="first"
        )
    )


    df_accept_all.to_csv(
        V2_ACCEPTED_PATH,
        index=False
    )


    # Save model
    joblib.dump(
        candidate_classifier,
        V2_CURRENT_MODEL_PATH
    )


    joblib.dump(
        candidate_classifier,
        V2_BEST_MODEL_PATH
    )


    with open(
        V2_BEST_METRICS_PATH,
        "w"
    ) as f:

        json.dump(
            candidate_metrics,
            f,
            indent=2
        )


    print(
        "New Macro F1:",
        round(
            candidate_macro_f1,
            4
        )
    )


    print(
        "Improvement:",
        round(
            candidate_macro_f1
            -
            best_macro_f1,
            4
        )
    )


# =========================================================
# REJECT / ROLLBACK
# =========================================================

else:

    print()
    print("=" * 65)
    print("CANDIDATE REJECTED ❌")
    print("=" * 65)

    print(
        "Reason:",
        decision_reason
    )


    if len(
        df_candidate_batch
    ) > 0:

        df_reject = (
            df_candidate_batch
            .copy()
        )


        df_reject[
            "rejected_at"
        ] = (
            pd.Timestamp.now()
        )


        df_reject[
            "reason"
        ] = (
            decision_reason
        )


        df_reject[
            "target_boundary"
        ] = (
            tuning_target[
                "confusing_with"
            ]
            if tuning_target
            else None
        )


        df_reject[
            "candidate_macro_f1"
        ] = (
            candidate_macro_f1
        )


        df_reject[
            "best_macro_f1"
        ] = (
            best_macro_f1
        )


        if os.path.exists(
            V2_REJECTED_PATH
        ):

            df_old_reject = pd.read_csv(
                V2_REJECTED_PATH
            )


            df_reject_all = pd.concat(
                [
                    df_old_reject,
                    df_reject,
                ],
                ignore_index=True
            )

        else:

            df_reject_all = (
                df_reject
            )


        df_reject_all.to_csv(
            V2_REJECTED_PATH,
            index=False
        )


    print(
        "Active dataset TIDAK diubah."
    )

    print(
        "Best model TIDAK diubah."
    )


print("=" * 65)


CANDIDATE REJECTED ❌
Reason: macro_f1_not_improved;class_regression;
Active dataset TIDAK diubah.
Best model TIDAK diubah.


In [22]:
# =========================================================
# CELL 20 — AUTO TUNE LOOP V2
# =========================================================

MAX_TUNE_ATTEMPTS = 12


def append_history_row(row):

    df_new = pd.DataFrame([row])

    if os.path.exists(V2_HISTORY_PATH):

        df_old = pd.read_csv(
            V2_HISTORY_PATH
        )

        df_all = pd.concat(
            [
                df_old,
                df_new,
            ],
            ignore_index=True
        )

    else:

        df_all = df_new

    df_all.to_csv(
        V2_HISTORY_PATH,
        index=False
    )


def get_current_active_dataset():

    if os.path.exists(
        V2_ACTIVE_DATASET_PATH
    ):

        return pd.read_csv(
            V2_ACTIVE_DATASET_PATH
        )

    return df_v2_train_baseline.copy()


def load_best_metrics():

    if not os.path.exists(
        V2_BEST_METRICS_PATH
    ):

        raise FileNotFoundError(
            V2_BEST_METRICS_PATH
        )

    with open(
        V2_BEST_METRICS_PATH,
        "r"
    ) as f:

        return json.load(f)


def load_best_classifier():

    if not os.path.exists(
        V2_BEST_MODEL_PATH
    ):

        raise FileNotFoundError(
            V2_BEST_MODEL_PATH
        )

    return joblib.load(
        V2_BEST_MODEL_PATH
    )


def get_confusion_ranking_from_best():

    df_active = get_current_active_dataset()

    classifier = load_best_classifier()

    # Evaluate ulang BEST pada fixed validation
    X_val = embedding_model.encode(
        df_v2_validation[
            "prompt"
        ].tolist(),
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True
    )

    y_true = (
        df_v2_validation[
            "label"
        ]
        .to_numpy()
    )

    y_pred = classifier.predict(
        X_val
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=LABELS
    )

    report = classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        output_dict=True,
        zero_division=0
    )

    metrics = {
        "per_class_recall": {
            label: float(
                report.get(
                    label,
                    {}
                ).get(
                    "recall",
                    0
                )
            )
            for label in LABELS
        }
    }

    evaluation = {
        "confusion_matrix": cm
    }

    return detect_biggest_confusion(
        evaluation,
        metrics
    )


def choose_boundary_candidates(
    confusion_ranking
):

    boundaries = []

    for item in confusion_ranking:

        true_label = item[
            "true_label"
        ]

        top_confusion = item[
            "top_confusion"
        ]

        if top_confusion is None:
            continue

        confusing_with = (
            top_confusion[
                "predicted_label"
            ]
        )

        boundaries.append({
            "target_label":
                true_label,

            "confusing_with":
                confusing_with,

            "current_recall":
                item[
                    "recall"
                ],

            "wrong_count":
                top_confusion[
                    "count"
                ],
        })

    return boundaries


def evaluate_class_regression(
    old_metrics,
    new_metrics
):

    regressions = []

    old_recalls = (
        old_metrics[
            "per_class_recall"
        ]
    )

    new_recalls = (
        new_metrics[
            "per_class_recall"
        ]
    )

    for label in LABELS:

        old_value = float(
            old_recalls.get(
                label,
                0
            )
        )

        new_value = float(
            new_recalls.get(
                label,
                0
            )
        )

        drop = (
            old_value
            - new_value
        )

        if drop > MAX_CLASS_RECALL_DROP:

            regressions.append({
                "label":
                    label,

                "old":
                    old_value,

                "new":
                    new_value,

                "drop":
                    drop,
            })

    return regressions


def save_rejected_batch(
    df_candidate_batch,
    target_label,
    confusing_with,
    reason,
    candidate_macro_f1,
    best_macro_f1
):

    if len(
        df_candidate_batch
    ) == 0:

        return

    df_reject = (
        df_candidate_batch
        .copy()
    )

    df_reject[
        "rejected_at"
    ] = pd.Timestamp.now()

    df_reject[
        "reason"
    ] = reason

    df_reject[
        "target_label"
    ] = target_label

    df_reject[
        "target_boundary"
    ] = confusing_with

    df_reject[
        "candidate_macro_f1"
    ] = candidate_macro_f1

    df_reject[
        "best_macro_f1"
    ] = best_macro_f1

    if os.path.exists(
        V2_REJECTED_PATH
    ):

        df_old = pd.read_csv(
            V2_REJECTED_PATH
        )

        df_all = pd.concat(
            [
                df_old,
                df_reject,
            ],
            ignore_index=True
        )

    else:

        df_all = df_reject

    df_all.to_csv(
        V2_REJECTED_PATH,
        index=False
    )


def save_accepted_batch(
    df_candidate_batch,
    target_label,
    confusing_with
):

    if len(
        df_candidate_batch
    ) == 0:

        return

    df_accept = (
        df_candidate_batch
        .copy()
    )

    df_accept[
        "accepted_at"
    ] = pd.Timestamp.now()

    df_accept[
        "target_label"
    ] = target_label

    df_accept[
        "target_boundary"
    ] = confusing_with

    if os.path.exists(
        V2_ACCEPTED_PATH
    ):

        df_old = pd.read_csv(
            V2_ACCEPTED_PATH
        )

        df_all = pd.concat(
            [
                df_old,
                df_accept,
            ],
            ignore_index=True
        )

    else:

        df_all = df_accept

    df_all = (
        df_all
        .drop_duplicates(
            subset=["prompt"],
            keep="first"
        )
        .reset_index(
            drop=True
        )
    )

    df_all.to_csv(
        V2_ACCEPTED_PATH,
        index=False
    )


def auto_tune_v2(
    max_attempts=MAX_TUNE_ATTEMPTS
):

    print()
    print("=" * 70)
    print("AUTO TUNE V2 START")
    print("=" * 70)

    no_improve_count = 0
    attempt = 0

    while attempt < max_attempts:

        attempt += 1

        print()
        print("=" * 70)
        print(
            f"ATTEMPT {attempt}/{max_attempts}"
        )
        print("=" * 70)

        # =================================================
        # LOAD CURRENT BEST
        # =================================================

        best_metrics = load_best_metrics()

        best_macro_f1 = float(
            best_metrics[
                "macro_f1"
            ]
        )

        print(
            "Current BEST Macro F1:",
            round(
                best_macro_f1,
                4
            )
        )

        # =================================================
        # FIND CURRENT CONFUSIONS
        # =================================================

        confusion_ranking = (
            get_confusion_ranking_from_best()
        )

        boundary_candidates = (
            choose_boundary_candidates(
                confusion_ranking
            )
        )

        if not boundary_candidates:

            print(
                "Tidak ada confusion boundary lagi."
            )
            break

        # Prioritaskan weakest class
        target = boundary_candidates[0]

        target_label = (
            target[
                "target_label"
            ]
        )

        confusing_with = (
            target[
                "confusing_with"
            ]
        )

        print()
        print(
            "Target:",
            target_label
        )

        print(
            "Boundary:",
            target_label,
            "<->",
            confusing_with
        )

        print(
            "Current recall:",
            round(
                float(
                    target[
                        "current_recall"
                    ]
                ),
                4
            )
        )

        # =================================================
        # GENERATE
        # =================================================

        try:

            candidate_prompts = (
                generate_candidate_prompts(
                    target_label=
                        target_label,

                    confusing_with=
                        confusing_with,

                    count=
                        CANDIDATE_BATCH_SIZE
                )
            )

        except Exception as e:

            print(
                "Generation failed:",
                repr(e)
            )

            no_improve_count += 1

            continue

        # =================================================
        # REMOVE DUPLICATES AGAINST ACTIVE DATA
        # =================================================

        df_active = (
            get_current_active_dataset()
        )

        existing = set(
            df_active[
                "prompt"
            ]
            .astype(str)
            .str.strip()
            .str.casefold()
        )

        unique_prompts = []

        seen = set()

        for prompt in candidate_prompts:

            clean = (
                str(prompt)
                .strip()
            )

            norm = (
                clean.casefold()
            )

            if not clean:
                continue

            if norm in existing:
                continue

            if norm in seen:
                continue

            seen.add(norm)

            unique_prompts.append(
                clean
            )

        candidate_prompts = (
            unique_prompts
        )

        print(
            "Unique candidates:",
            len(
                candidate_prompts
            )
        )

        if len(
            candidate_prompts
        ) == 0:

            print(
                "Tidak ada candidate unik."
            )

            no_improve_count += 1

            continue

        # =================================================
        # TEACHER VALIDATION
        # =================================================

        validation_results = (
            validate_candidate_batch(
                prompts=
                    candidate_prompts,

                expected_label=
                    target_label
            )
        )

        accepted_results = [
            item
            for item
            in validation_results
            if item[
                "status"
            ] == "ok"
        ]

        print(
            "Teacher accepted:",
            len(
                accepted_results
            ),
            "/",
            len(
                candidate_prompts
            )
        )

        if len(
            accepted_results
        ) == 0:

            print(
                "Tidak ada candidate lolos teacher."
            )

            no_improve_count += 1

            continue

        # =================================================
        # BUILD CANDIDATE DF
        # =================================================

        df_candidate_batch = (
            build_candidate_dataframe(
                accepted_results
            )
        )

        df_candidate_train = pd.concat(
            [
                df_active,
                df_candidate_batch,
            ],
            ignore_index=True
        )

        df_candidate_train = (
            df_candidate_train
            .drop_duplicates(
                subset=["prompt"],
                keep="first"
            )
            .reset_index(
                drop=True
            )
        )

        # =================================================
        # TRAIN CANDIDATE
        # =================================================

        (
            candidate_classifier,
            candidate_metrics,
            candidate_evaluation,
        ) = train_and_evaluate_router(
            df_candidate_train,
            df_v2_validation
        )

        candidate_macro_f1 = float(
            candidate_metrics[
                "macro_f1"
            ]
        )

        improvement = (
            candidate_macro_f1
            - best_macro_f1
        )

        # =================================================
        # TARGET RECALL
        # =================================================

        old_target_recall = float(
            best_metrics[
                "per_class_recall"
            ].get(
                target_label,
                0
            )
        )

        new_target_recall = float(
            candidate_metrics[
                "per_class_recall"
            ].get(
                target_label,
                0
            )
        )

        # =================================================
        # REGRESSION CHECK
        # =================================================

        regressions = (
            evaluate_class_regression(
                best_metrics,
                candidate_metrics
            )
        )

        print()
        print("-" * 70)

        print(
            "Best Macro F1      :",
            round(
                best_macro_f1,
                4
            )
        )

        print(
            "Candidate Macro F1 :",
            round(
                candidate_macro_f1,
                4
            )
        )

        print(
            "Improvement        :",
            round(
                improvement,
                4
            )
        )

        print(
            "Target recall      :",
            round(
                old_target_recall,
                4
            ),
            "->",
            round(
                new_target_recall,
                4
            )
        )

        # =================================================
        # SHOW REGRESSIONS
        # =================================================

        if regressions:

            print()
            print(
                "Class regressions:"
            )

            for item in regressions:

                print(
                    f"  {item['label']:16} "
                    f"{item['old']:.3f} "
                    f"-> {item['new']:.3f} "
                    f"(drop {item['drop']:.3f})"
                )

        else:

            print(
                "Class regressions : none"
            )

        # =================================================
        # QUALITY GATE
        # =================================================

        f1_pass = (
            improvement
            >= MIN_IMPROVEMENT
        )

        target_recall_pass = (
            new_target_recall
            >= old_target_recall
        )

        regression_pass = (
            len(
                regressions
            ) == 0
        )

        accept = (
            f1_pass
            and
            target_recall_pass
            and
            regression_pass
        )

        # =================================================
        # ACCEPT
        # =================================================

        if accept:

            print()
            print(
                "CANDIDATE ACCEPTED ✅"
            )

            df_candidate_train.to_csv(
                V2_ACTIVE_DATASET_PATH,
                index=False
            )

            save_accepted_batch(
                df_candidate_batch,
                target_label,
                confusing_with
            )

            joblib.dump(
                candidate_classifier,
                V2_CURRENT_MODEL_PATH
            )

            joblib.dump(
                candidate_classifier,
                V2_BEST_MODEL_PATH
            )

            with open(
                V2_BEST_METRICS_PATH,
                "w"
            ) as f:

                json.dump(
                    candidate_metrics,
                    f,
                    indent=2
                )

            decision = "accepted"

            reason = "improved"

            no_improve_count = 0

        # =================================================
        # REJECT
        # =================================================

        else:

            print()
            print(
                "CANDIDATE REJECTED ❌"
            )

            reasons = []

            if not f1_pass:

                reasons.append(
                    "macro_f1_not_improved"
                )

            if not target_recall_pass:

                reasons.append(
                    "target_recall_dropped"
                )

            if not regression_pass:

                reasons.append(
                    "class_regression"
                )

            reason = ";".join(
                reasons
            )

            print(
                "Reason:",
                reason
            )

            save_rejected_batch(
                df_candidate_batch=
                    df_candidate_batch,

                target_label=
                    target_label,

                confusing_with=
                    confusing_with,

                reason=
                    reason,

                candidate_macro_f1=
                    candidate_macro_f1,

                best_macro_f1=
                    best_macro_f1
            )

            decision = "rejected"

            no_improve_count += 1

        # =================================================
        # HISTORY
        # =================================================

        append_history_row({
            "timestamp":
                pd.Timestamp.now(),

            "attempt":
                attempt,

            "target_label":
                target_label,

            "confusing_with":
                confusing_with,

            "batch_size":
                len(
                    df_candidate_batch
                ),

            "best_macro_f1_before":
                best_macro_f1,

            "candidate_macro_f1":
                candidate_macro_f1,

            "improvement":
                improvement,

            "target_recall_before":
                old_target_recall,

            "target_recall_after":
                new_target_recall,

            "regression_count":
                len(
                    regressions
                ),

            "decision":
                decision,

            "reason":
                reason,
        })

        # =================================================
        # EARLY STOP
        # =================================================

        print()
        print(
            "No-improve streak:",
            no_improve_count,
            "/",
            MAX_NO_IMPROVE_ROUNDS
        )

        if (
            no_improve_count
            >= MAX_NO_IMPROVE_ROUNDS
        ):

            print()
            print(
                "EARLY STOP: terlalu banyak "
                "candidate berturut-turut tidak improve."
            )

            break

    # =====================================================
    # FINAL
    # =====================================================

    final_metrics = (
        load_best_metrics()
    )

    print()
    print("=" * 70)
    print("AUTO TUNE V2 FINISHED")
    print("=" * 70)

    print(
        "Best Macro F1:",
        round(
            float(
                final_metrics[
                    "macro_f1"
                ]
            ),
            4
        )
    )

    print(
        "Best Accuracy:",
        round(
            float(
                final_metrics[
                    "accuracy"
                ]
            ),
            4
        )
    )

    print(
        "Active dataset:",
        len(
            get_current_active_dataset()
        )
    )

    print(
        "History:",
        V2_HISTORY_PATH
    )

    print("=" * 70)

    return final_metrics

In [23]:
# =========================================================
# CELL 21 — RUN AUTO TUNE V2
# =========================================================

final_v2_metrics = (
    auto_tune_v2(
        max_attempts=12
    )
)


AUTO TUNE V2 START

ATTEMPT 1/12
Current BEST Macro F1: 0.7036

Target: GENERAL
Boundary: GENERAL <-> SIMPLE
Current recall: 0.4375
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (7.59s)
Unique candidates: 6


Validate GENERAL:   0%|          | 0/6 [00:00<?, ?it/s]

  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (4.35s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (9.43s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (3.91s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (3.83s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (59.52s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (3.01s)
Teacher accepted: 6 / 6

Generating TRAIN embeddings...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


Generating VALIDATION embeddings...


...

  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (4.82s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (2.51s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (6.71s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (6.43s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (10.12s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (6.63s)
Teacher accepted: 4 / 6

Generating TRAIN embeddings...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


Generating VALIDATION embeddings...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Training distribution:
SIMPLE            64
GENERAL           69
REASONING         91
CODING_SIMPLE     41
CODING_COMPLEX    45
TRANSFORM         40
CREATIVE          32
Name: count, dtype: int64

Searching best C...

----------------------------------------------------------------------
Best Macro F1      : 0.7036
Candidate Macro F1 : 0.692
Improvement        : -0.0116
Target recall      : 0.4375 -> 0.375

Class regressions:
  GENERAL          0.438 -> 0.375 (drop 0.062)

CANDIDATE REJECTED ❌
Reason: macro_f1_not_improved;target_recall_dropped;class_regression

No-improve streak: 4 / 4

EARLY STOP: terlalu banyak candidate berturut-turut tidak improve.

AUTO TUNE V2 FINISHED
Best Macro F1: 0.7036
Best Accuracy: 0.6947
Active dataset: 378
History: /content/drive/MyDrive/router_classifier/router_v2_history.csv


In [24]:
# =========================================================
# CELL 22 — ERROR INSPECTOR
# ROUTER AUTO TUNE V2
# =========================================================

best_classifier = joblib.load(
    V2_BEST_MODEL_PATH
)

X_val = embedding_model.encode(
    df_v2_validation["prompt"].tolist(),
    batch_size=64,
    show_progress_bar=False,
    normalize_embeddings=True
)

y_true = (
    df_v2_validation["label"]
    .to_numpy()
)

y_pred = (
    best_classifier.predict(
        X_val
    )
)

if hasattr(
    best_classifier,
    "predict_proba"
):

    probabilities = (
        best_classifier.predict_proba(
            X_val
        )
    )

else:

    probabilities = None


error_rows = []


for i, (
    prompt,
    true_label,
    predicted_label
) in enumerate(
    zip(
        df_v2_validation["prompt"],
        y_true,
        y_pred
    )
):

    if true_label == predicted_label:
        continue

    row = {
        "prompt": prompt,
        "true_label": true_label,
        "predicted_label": predicted_label,
    }

    if probabilities is not None:

        probs = probabilities[i]

        pred_idx = list(
            best_classifier.classes_
        ).index(
            predicted_label
        )

        true_idx = list(
            best_classifier.classes_
        ).index(
            true_label
        )

        row[
            "predicted_confidence"
        ] = float(
            probs[pred_idx]
        )

        row[
            "true_score"
        ] = float(
            probs[true_idx]
        )

        row[
            "margin"
        ] = float(
            probs[pred_idx]
            -
            probs[true_idx]
        )

    error_rows.append(
        row
    )


df_errors = pd.DataFrame(
    error_rows
)


print()
print("=" * 70)
print("VALIDATION ERRORS")
print("=" * 70)

print(
    "Total validation:",
    len(
        df_v2_validation
    )
)

print(
    "Total errors:",
    len(
        df_errors
    )
)

print()


# =========================================================
# CONFUSION PAIRS
# =========================================================

confusion_pairs = (
    df_errors
    .groupby(
        [
            "true_label",
            "predicted_label"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
)


display(
    confusion_pairs
)


# =========================================================
# SHOW MOST IMPORTANT ERRORS
# =========================================================

if (
    "margin"
    in df_errors.columns
):

    df_errors_sorted = (
        df_errors
        .sort_values(
            [
                "true_label",
                "margin"
            ],
            ascending=[
                True,
                False
            ]
        )
    )

else:

    df_errors_sorted = (
        df_errors.copy()
    )


display(
    df_errors_sorted
)


print("=" * 70)


VALIDATION ERRORS
Total validation: 95
Total errors: 29



,true_label,predicted_label,count
8,REASONING,CODING_COMPLEX,7
7,GENERAL,SIMPLE,3
4,GENERAL,CODING_COMPLEX,3
6,GENERAL,REASONING,2
9,REASONING,GENERAL,2
11,SIMPLE,CREATIVE,2
3,CREATIVE,TRANSFORM,1
0,CODING_COMPLEX,REASONING,1
5,GENERAL,CODING_SIMPLE,1
1,CODING_SIMPLE,REASONING,1


,prompt,true_label,predicted_label,predicted_confidence,true_score,margin
26,Implementasikan solusi caching terdistribusi m...,CODING_COMPLEX,REASONING,0.488209,0.298160,0.190049
4,Buatlah grafik yang membandingkan biaya Undang...,CODING_SIMPLE,REASONING,0.310980,0.081661,0.229319
28,Tulis pertanyaan untuk menilai pemahaman suatu...,CREATIVE,GENERAL,0.358152,0.156720,0.201432
24,Berikan contoh postingan media sosial yang men...,CREATIVE,TRANSFORM,0.404007,0.212383,0.191625
18,Saya menggunakan PostgreSQL untuk backend apli...,GENERAL,REASONING,0.541233,0.053992,0.487241
16,Apa pattern terbaik untuk circuit breaker di m...,GENERAL,CODING_COMPLEX,0.466400,0.095385,0.371015
0,Saya butuh saran untuk scaling microservice di...,GENERAL,REASONING,0.462013,0.101155,0.360858
25,How to implement secure authentication using O...,GENERAL,CODING_COMPLEX,0.480614,0.195024,0.285590
15,Apa manfaat olahraga teratur untuk kesehatan t...,GENERAL,SIMPLE,0.516665,0.266744,0.249921
11,Bagaimana cara membuat laporan keuangan sederh...,GENERAL,CODING_SIMPLE,0.401212,0.153804,0.247408


In [33]:
# =========================================================
# CELL 23 — AUTOMATIC RELABEL AUDIT
# ROUTER AUTO TUNE V2
# =========================================================

def audit_validation_errors(
    df_errors,
    classifier,
    max_rows=None
):

    if df_errors is None or len(df_errors) == 0:
        print("Tidak ada validation error untuk diaudit.")
        return pd.DataFrame()

    audit_source = (
        df_errors.copy()
        .reset_index(drop=True)
    )

    if max_rows is not None:
        audit_source = audit_source.head(
            int(max_rows)
        )

    rows = []

    print()
    print("=" * 70)
    print("VALIDATION RELABEL AUDIT")
    print("=" * 70)

    for idx, row in tqdm(
        audit_source.iterrows(),
        total=len(audit_source),
        desc="Teacher relabel audit"
    ):

        prompt = str(
            row["prompt"]
        ).strip()

        old_label = str(
            row["true_label"]
        ).strip()

        model_label = str(
            row["predicted_label"]
        ).strip()

        # -----------------------------------------
        # Ask teacher V2
        # -----------------------------------------

        teacher_label = None
        teacher_confidence = None
        teacher_difficulty = None
        teacher_status = "error"

        last_error = None

        for attempt in range(3):

            try:

                raw = call_teacher(
                    prompt
                )

                parsed = parse_teacher_output(
                    raw
                )

                teacher_label = (
                    parsed["label"]
                )

                teacher_confidence = float(
                    parsed["confidence"]
                )

                teacher_difficulty = int(
                    parsed["difficulty"]
                )

                teacher_status = "ok"

                break

            except Exception as e:

                last_error = e

                if attempt < 2:
                    time.sleep(
                        2 * (attempt + 1)
                    )

        # -----------------------------------------
        # Decide suggested action
        # -----------------------------------------

        if teacher_status != "ok":

            suggested_action = (
                "REVIEW_MANUAL"
            )

        elif (
            teacher_label == model_label
            and
            teacher_label != old_label
            and
            teacher_confidence >= MIN_CONFIDENCE
        ):

            suggested_action = (
                "LIKELY_OLD_LABEL_WRONG"
            )

        elif (
            teacher_label == old_label
            and
            teacher_label != model_label
        ):

            suggested_action = (
                "MODEL_ERROR"
            )

        elif (
            teacher_label == old_label
            and
            teacher_label == model_label
        ):

            suggested_action = (
                "CONSISTENT"
            )

        else:

            suggested_action = (
                "AMBIGUOUS_REVIEW"
            )

        rows.append({
            "prompt":
                prompt,

            "old_label":
                old_label,

            "teacher_label":
                teacher_label,

            "model_label":
                model_label,

            "teacher_confidence":
                teacher_confidence,

            "teacher_difficulty":
                teacher_difficulty,

            "teacher_status":
                teacher_status,

            "suggested_action":
                suggested_action,

            "model_predicted_confidence":
                row.get(
                    "predicted_confidence",
                    np.nan
                ),

            "model_true_score":
                row.get(
                    "true_score",
                    np.nan
                ),

            "model_margin":
                row.get(
                    "margin",
                    np.nan
                ),

            "error":
                (
                    str(last_error)
                    if teacher_status != "ok"
                    else None
                ),
        })

    return pd.DataFrame(
        rows
    )


# =========================================================
# RUN AUDIT
# =========================================================

best_classifier = joblib.load(
    V2_BEST_MODEL_PATH
)

df_relabel_audit = (
    audit_validation_errors(
        df_errors=df_errors,
        classifier=best_classifier
    )
)


# =========================================================
# DISPLAY SUMMARY
# =========================================================

print()
print("=" * 70)
print("AUDIT SUMMARY")
print("=" * 70)

print(
    "Audited errors:",
    len(
        df_relabel_audit
    )
)

print()

if len(
    df_relabel_audit
) > 0:

    action_counts = (
        df_relabel_audit[
            "suggested_action"
        ]
        .value_counts()
    )

    print(
        action_counts
    )

print("=" * 70)


# =========================================================
# DISPLAY FULL TABLE
# =========================================================

display(
    df_relabel_audit
)


VALIDATION RELABEL AUDIT


Teacher relabel audit:   0%|          | 0/29 [00:00<?, ?it/s]

  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.94s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.98s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.55s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.58s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.90s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.96s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (1.00s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.83s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.98s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: 

,prompt,old_label,teacher_label,model_label,teacher_confidence,teacher_difficulty,teacher_status,suggested_action,model_predicted_confidence,model_true_score,model_margin,error
0,Saya butuh saran untuk scaling microservice di...,GENERAL,REASONING,REASONING,0.90,4,ok,LIKELY_OLD_LABEL_WRONG,0.462013,0.101155,0.360858,None
1,"Eja kata ""gambang"".",SIMPLE,SIMPLE,CREATIVE,0.95,1,ok,MODEL_ERROR,0.494140,0.306022,0.188118,None
2,Saya ingin merancang arsitektur microservices ...,REASONING,REASONING,GENERAL,0.90,4,ok,MODEL_ERROR,0.409206,0.267866,0.141340,None
3,Saya perlu melakukan security audit terhadap i...,REASONING,CODING_COMPLEX,CODING_COMPLEX,0.95,4,ok,LIKELY_OLD_LABEL_WRONG,0.471891,0.306862,0.165029,None
4,Buatlah grafik yang membandingkan biaya Undang...,CODING_SIMPLE,REASONING,REASONING,0.85,4,ok,LIKELY_OLD_LABEL_WRONG,0.310980,0.081661,0.229319,None
5,Jelaskan mengapa pernyataan yang diberikan mun...,REASONING,GENERAL,TRANSFORM,0.90,2,ok,AMBIGUOUS_REVIEW,0.481352,0.082510,0.398842,None
6,Design a scalable authentication service using...,REASONING,CODING_COMPLEX,CODING_COMPLEX,0.95,5,ok,LIKELY_OLD_LABEL_WRONG,0.390159,0.375373,0.014786,None
7,Desain API GraphQL yang efisien untuk menguran...,REASONING,CODING_COMPLEX,CODING_COMPLEX,0.90,4,ok,LIKELY_OLD_LABEL_WRONG,0.625061,0.252761,0.372300,None
8,Profiling aplikasi Node.js menunjukkan beban C...,REASONING,REASONING,CODING_COMPLEX,0.90,4,ok,MODEL_ERROR,0.631803,0.221914,0.409889,None
9,Apa perbedaan antara laptop dan tablet?,SIMPLE,GENERAL,GENERAL,0.90,2,ok,LIKELY_OLD_LABEL_WRONG,0.418409,0.337421,0.080988,None


In [34]:
# =========================================================
# CELL 24 — LIKELY WRONG LABELS
# =========================================================

df_likely_wrong = (
    df_relabel_audit[
        df_relabel_audit[
            "suggested_action"
        ]
        ==
        "LIKELY_OLD_LABEL_WRONG"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print()
print("=" * 70)
print("LIKELY WRONG OLD LABELS")
print("=" * 70)

print(
    "Count:",
    len(
        df_likely_wrong
    )
)

print()


if len(
    df_likely_wrong
) > 0:

    display(
        df_likely_wrong[
            [
                "prompt",
                "old_label",
                "teacher_label",
                "model_label",
                "teacher_confidence",
                "model_predicted_confidence",
                "model_margin",
            ]
        ]
    )

else:

    print(
        "Tidak ada strong relabel candidate."
    )


print("=" * 70)


LIKELY WRONG OLD LABELS
Count: 12



,prompt,old_label,teacher_label,model_label,teacher_confidence,model_predicted_confidence,model_margin
0,Saya butuh saran untuk scaling microservice di...,GENERAL,REASONING,REASONING,0.90,0.462013,0.360858
1,Saya perlu melakukan security audit terhadap i...,REASONING,CODING_COMPLEX,CODING_COMPLEX,0.95,0.471891,0.165029
2,Buatlah grafik yang membandingkan biaya Undang...,CODING_SIMPLE,REASONING,REASONING,0.85,0.310980,0.229319
3,Design a scalable authentication service using...,REASONING,CODING_COMPLEX,CODING_COMPLEX,0.95,0.390159,0.014786
4,Desain API GraphQL yang efisien untuk menguran...,REASONING,CODING_COMPLEX,CODING_COMPLEX,0.90,0.625061,0.372300
5,Apa perbedaan antara laptop dan tablet?,SIMPLE,GENERAL,GENERAL,0.90,0.418409,0.080988
6,Bagaimana mengatasi race condition pada thread...,REASONING,CODING_COMPLEX,CODING_COMPLEX,0.95,0.731051,0.595410
7,Bagaimana merancang ulang arsitektur microserv...,REASONING,CODING_COMPLEX,CODING_COMPLEX,0.95,0.462361,0.215347
8,Bagaimana cara melakukan code review untuk men...,GENERAL,CODING_COMPLEX,CODING_COMPLEX,0.95,0.308407,0.161929
9,Buat daftar FAQ dari pertanyaan-pertanyaan ber...,SIMPLE,TRANSFORM,TRANSFORM,0.90,0.313442,0.122711


In [35]:
# =========================================================
# CELL 25 — BUILD CORRECTED VALIDATION CANDIDATE
# ROUTER AUTO TUNE V2
# =========================================================

df_validation_corrected = (
    df_v2_validation
    .copy()
)


# =========================================================
# STRONG RELABEL CANDIDATES
# =========================================================

df_safe_relabel = (
    df_relabel_audit[
        (
            df_relabel_audit[
                "suggested_action"
            ]
            == "LIKELY_OLD_LABEL_WRONG"
        )
        &
        (
            df_relabel_audit[
                "teacher_confidence"
            ]
            >= MIN_CONFIDENCE
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print()
print("=" * 70)
print("CORRECTED VALIDATION CANDIDATE")
print("=" * 70)

print(
    "Relabel candidates:",
    len(
        df_safe_relabel
    )
)


# =========================================================
# APPLY ONLY TO COPY
# =========================================================

changes = []


for _, row in df_safe_relabel.iterrows():

    prompt = str(
        row["prompt"]
    ).strip()

    old_label = str(
        row["old_label"]
    )

    new_label = str(
        row["teacher_label"]
    )


    mask = (
        df_validation_corrected[
            "prompt"
        ]
        .astype(str)
        .str.strip()
        ==
        prompt
    )


    matches = int(
        mask.sum()
    )


    if matches == 0:

        continue


    current_label = (
        df_validation_corrected
        .loc[
            mask,
            "label"
        ]
        .iloc[0]
    )


    df_validation_corrected.loc[
        mask,
        "label"
    ] = new_label


    changes.append({
        "prompt":
            prompt,

        "old_label":
            current_label,

        "new_label":
            new_label,

        "teacher_confidence":
            float(
                row[
                    "teacher_confidence"
                ]
            ),
    })


df_validation_changes = (
    pd.DataFrame(
        changes
    )
)


print(
    "Applied changes:",
    len(
        df_validation_changes
    )
)

print()

display(
    df_validation_changes
)

print("=" * 70)


CORRECTED VALIDATION CANDIDATE
Relabel candidates: 12
Applied changes: 12



,prompt,old_label,new_label,teacher_confidence
0,Saya butuh saran untuk scaling microservice di...,GENERAL,REASONING,0.90
1,Saya perlu melakukan security audit terhadap i...,REASONING,CODING_COMPLEX,0.95
2,Buatlah grafik yang membandingkan biaya Undang...,CODING_SIMPLE,REASONING,0.85
3,Design a scalable authentication service using...,REASONING,CODING_COMPLEX,0.95
4,Desain API GraphQL yang efisien untuk menguran...,REASONING,CODING_COMPLEX,0.90
5,Apa perbedaan antara laptop dan tablet?,SIMPLE,GENERAL,0.90
6,Bagaimana mengatasi race condition pada thread...,REASONING,CODING_COMPLEX,0.95
7,Bagaimana merancang ulang arsitektur microserv...,REASONING,CODING_COMPLEX,0.95
8,Bagaimana cara melakukan code review untuk men...,GENERAL,CODING_COMPLEX,0.95
9,Buat daftar FAQ dari pertanyaan-pertanyaan ber...,SIMPLE,TRANSFORM,0.90


In [36]:
# =========================================================
# CELL 26 — RE-EVALUATE BEST MODEL
# USING CORRECTED VALIDATION CANDIDATE
# =========================================================

best_classifier = joblib.load(
    V2_BEST_MODEL_PATH
)


# =========================================================
# EMBED CORRECTED VALIDATION
# =========================================================

X_corrected_val = (
    embedding_model.encode(
        df_validation_corrected[
            "prompt"
        ].tolist(),
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True
    )
)


y_corrected_true = (
    df_validation_corrected[
        "label"
    ]
    .to_numpy()
)


y_corrected_pred = (
    best_classifier.predict(
        X_corrected_val
    )
)


# =========================================================
# NEW METRICS
# =========================================================

corrected_accuracy = (
    accuracy_score(
        y_corrected_true,
        y_corrected_pred
    )
)


corrected_macro_f1 = (
    f1_score(
        y_corrected_true,
        y_corrected_pred,
        average="macro",
        zero_division=0
    )
)


# =========================================================
# OLD METRICS
# =========================================================

with open(
    V2_BEST_METRICS_PATH,
    "r"
) as f:

    old_best_metrics = json.load(
        f
    )


old_accuracy = float(
    old_best_metrics[
        "accuracy"
    ]
)


old_macro_f1 = float(
    old_best_metrics[
        "macro_f1"
    ]
)


# =========================================================
# DIFFERENCE
# =========================================================

accuracy_change = (
    corrected_accuracy
    - old_accuracy
)


f1_change = (
    corrected_macro_f1
    - old_macro_f1
)


# =========================================================
# DISPLAY
# =========================================================

print()
print("=" * 70)
print("VALIDATION BEFORE VS CORRECTED")
print("=" * 70)

print(
    f"OLD Accuracy     : {old_accuracy:.4f}"
)

print(
    f"NEW Accuracy     : {corrected_accuracy:.4f}"
)

print(
    f"Accuracy change  : {accuracy_change:+.4f}"
)

print()

print(
    f"OLD Macro F1     : {old_macro_f1:.4f}"
)

print(
    f"NEW Macro F1     : {corrected_macro_f1:.4f}"
)

print(
    f"Macro F1 change  : {f1_change:+.4f}"
)

print()

print(
    "Relabelled items :",
    len(
        df_validation_changes
    )
)

print("=" * 70)


# =========================================================
# PER-CLASS REPORT
# =========================================================

corrected_report = (
    classification_report(
        y_corrected_true,
        y_corrected_pred,
        labels=LABELS,
        output_dict=True,
        zero_division=0
    )
)


print()
print("PER-CLASS RECALL AFTER CORRECTION")
print("-" * 70)

for label in LABELS:

    recall = float(
        corrected_report
        .get(
            label,
            {}
        )
        .get(
            "recall",
            0
        )
    )

    print(
        f"{label:16} {recall:.3f}"
    )


VALIDATION BEFORE VS CORRECTED
OLD Accuracy     : 0.6947
NEW Accuracy     : 0.8211
Accuracy change  : +0.1263

OLD Macro F1     : 0.7036
NEW Macro F1     : 0.8082
Macro F1 change  : +0.1046

Relabelled items : 12

PER-CLASS RECALL AFTER CORRECTION
----------------------------------------------------------------------
SIMPLE           0.857
GENERAL          0.571
REASONING        0.789
CODING_SIMPLE    1.000
CODING_COMPLEX   0.947
TRANSFORM        0.818
CREATIVE         0.750


In [37]:
V2_VALIDATION_CORRECTED_PATH = (
    f"{SAVE_DIR}/router_v2_validation_corrected.csv"
)

df_validation_corrected.to_csv(
    V2_VALIDATION_CORRECTED_PATH,
    index=False
)

print(
    "Saved:",
    V2_VALIDATION_CORRECTED_PATH
)

Saved: /content/drive/MyDrive/router_classifier/router_v2_validation_corrected.csv


In [38]:
# =========================================================
# CELL 27 — TRAINING DATASET LABEL AUDIT
# ROUTER AUTO TUNE V2
# =========================================================

AUDIT_LABELS = [
    "GENERAL",
    "REASONING",
    "CODING_COMPLEX",
]

MAX_AUDIT_PER_LABEL = 40


# =========================================================
# LOAD ACTIVE TRAINING DATA
# =========================================================

if os.path.exists(
    V2_ACTIVE_DATASET_PATH
):

    df_training_audit_source = pd.read_csv(
        V2_ACTIVE_DATASET_PATH
    )

else:

    df_training_audit_source = (
        df_v2_train_baseline.copy()
    )


# =========================================================
# FILTER TARGET LABELS
# =========================================================

audit_parts = []

for label in AUDIT_LABELS:

    part = (
        df_training_audit_source[
            df_training_audit_source[
                "label"
            ] == label
        ]
        .copy()
    )

    if len(part) > MAX_AUDIT_PER_LABEL:

        part = part.sample(
            n=MAX_AUDIT_PER_LABEL,
            random_state=RANDOM_STATE
        )

    audit_parts.append(
        part
    )


df_training_audit_sample = (
    pd.concat(
        audit_parts,
        ignore_index=True
    )
    .drop_duplicates(
        subset=["prompt"]
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 70)
print("TRAINING AUDIT SAMPLE")
print("=" * 70)

print(
    "Samples:",
    len(
        df_training_audit_sample
    )
)

print()

print(
    df_training_audit_sample[
        "label"
    ].value_counts()
)

print("=" * 70)


# =========================================================
# TEACHER AUDIT
# =========================================================

training_audit_rows = []


for _, row in tqdm(
    df_training_audit_sample.iterrows(),
    total=len(
        df_training_audit_sample
    ),
    desc="Training label audit"
):

    prompt = str(
        row["prompt"]
    ).strip()

    old_label = str(
        row["label"]
    ).strip()

    teacher_label = None
    teacher_confidence = None
    teacher_difficulty = None
    teacher_status = "error"

    last_error = None


    for attempt in range(3):

        try:

            raw = call_teacher(
                prompt
            )

            parsed = parse_teacher_output(
                raw
            )

            teacher_label = (
                parsed["label"]
            )

            teacher_confidence = float(
                parsed["confidence"]
            )

            teacher_difficulty = int(
                parsed["difficulty"]
            )

            teacher_status = "ok"

            break

        except Exception as e:

            last_error = e

            if attempt < 2:

                time.sleep(
                    2 * (attempt + 1)
                )


    if teacher_status != "ok":

        action = "REVIEW_MANUAL"

    elif (
        teacher_label == old_label
    ):

        action = "KEEP"

    elif (
        teacher_confidence
        >= 0.90
    ):

        action = "STRONG_RELABEL_CANDIDATE"

    elif (
        teacher_confidence
        >= MIN_CONFIDENCE
    ):

        action = "REVIEW_RELABEL"

    else:

        action = "AMBIGUOUS"


    training_audit_rows.append({
        "prompt":
            prompt,

        "old_label":
            old_label,

        "teacher_label":
            teacher_label,

        "teacher_confidence":
            teacher_confidence,

        "teacher_difficulty":
            teacher_difficulty,

        "teacher_status":
            teacher_status,

        "suggested_action":
            action,

        "error":
            (
                str(last_error)
                if teacher_status != "ok"
                else None
            ),
    })


df_training_audit = pd.DataFrame(
    training_audit_rows
)


# =========================================================
# SUMMARY
# =========================================================

print()
print("=" * 70)
print("TRAINING AUDIT SUMMARY")
print("=" * 70)

print(
    df_training_audit[
        "suggested_action"
    ].value_counts()
)

print()

print(
    "Old -> Teacher label changes:"
)

change_summary = (
    df_training_audit[
        df_training_audit[
            "old_label"
        ]
        !=
        df_training_audit[
            "teacher_label"
        ]
    ]
    .groupby(
        [
            "old_label",
            "teacher_label"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
)

display(
    change_summary
)


# =========================================================
# STRONG CANDIDATES
# =========================================================

df_training_strong_relabel = (
    df_training_audit[
        df_training_audit[
            "suggested_action"
        ]
        ==
        "STRONG_RELABEL_CANDIDATE"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print()
print(
    "Strong relabel candidates:",
    len(
        df_training_strong_relabel
    )
)

display(
    df_training_strong_relabel
)


TRAINING AUDIT SAMPLE
Samples: 120

label
GENERAL           40
REASONING         40
CODING_COMPLEX    40
Name: count, dtype: int64


Training label audit:   0%|          | 0/120 [00:00<?, ?it/s]

  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.61s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.86s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.96s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.54s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.68s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.71s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.94s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.98s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.86s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: 

,old_label,teacher_label,count
4,REASONING,CODING_COMPLEX,22
3,GENERAL,REASONING,6
0,GENERAL,CODING_COMPLEX,5
2,GENERAL,CREATIVE,2
1,GENERAL,CODING_SIMPLE,1



Strong relabel candidates: 31


,prompt,old_label,teacher_label,teacher_confidence,teacher_difficulty,teacher_status,suggested_action,error
0,Berikan contoh respon yang menunjukkan perilak...,GENERAL,CREATIVE,0.90,2,ok,STRONG_RELABEL_CANDIDATE,None
1,Bagaimana mengatur concurrency control pada da...,GENERAL,REASONING,0.90,4,ok,STRONG_RELABEL_CANDIDATE,None
2,Jelaskan cara mendeteksi dan memperbaiki memor...,GENERAL,CODING_COMPLEX,0.95,4,ok,STRONG_RELABEL_CANDIDATE,None
3,I'm building a distributed event processing pi...,GENERAL,CODING_COMPLEX,0.95,5,ok,STRONG_RELABEL_CANDIDATE,None
4,Saya menggunakan Django REST Framework untuk b...,GENERAL,CODING_SIMPLE,0.90,2,ok,STRONG_RELABEL_CANDIDATE,None
5,What are best practices for securing a Docker ...,GENERAL,REASONING,0.90,3,ok,STRONG_RELABEL_CANDIDATE,None
6,Can you explain how to detect and eliminate de...,GENERAL,CODING_COMPLEX,0.90,4,ok,STRONG_RELABEL_CANDIDATE,None
7,Saya perlu merancang arsitektur mikrolayanan u...,GENERAL,CODING_COMPLEX,0.90,4,ok,STRONG_RELABEL_CANDIDATE,None
8,Explain how to design a fault‑tolerant distrib...,GENERAL,CODING_COMPLEX,0.90,4,ok,STRONG_RELABEL_CANDIDATE,None
9,Bagaimana cara mendeteksi dan memperbaiki race...,REASONING,CODING_COMPLEX,0.95,4,ok,STRONG_RELABEL_CANDIDATE,None


In [39]:
# =========================================================
# CELL 28 — BUILD CORRECTED TRAINING CANDIDATE
# ROUTER AUTO TUNE V2
# =========================================================

# Load active training dataset
if os.path.exists(
    V2_ACTIVE_DATASET_PATH
):

    df_train_before_relabel = pd.read_csv(
        V2_ACTIVE_DATASET_PATH
    )

else:

    df_train_before_relabel = (
        df_v2_train_baseline.copy()
    )


df_train_corrected_candidate = (
    df_train_before_relabel.copy()
)


# =========================================================
# APPLY STRONG RELABELS TO COPY ONLY
# =========================================================

training_changes = []


for _, row in (
    df_training_strong_relabel
    .iterrows()
):

    prompt = str(
        row["prompt"]
    ).strip()

    old_label = str(
        row["old_label"]
    ).strip()

    new_label = str(
        row["teacher_label"]
    ).strip()

    teacher_confidence = float(
        row["teacher_confidence"]
    )


    mask = (
        df_train_corrected_candidate[
            "prompt"
        ]
        .astype(str)
        .str.strip()
        ==
        prompt
    )


    if not mask.any():
        continue


    actual_old_label = (
        df_train_corrected_candidate
        .loc[
            mask,
            "label"
        ]
        .iloc[0]
    )


    # Safety:
    # hanya ubah kalau dataset masih memiliki old label
    if actual_old_label != old_label:
        continue


    df_train_corrected_candidate.loc[
        mask,
        "label"
    ] = new_label


    training_changes.append({
        "prompt":
            prompt,

        "old_label":
            old_label,

        "new_label":
            new_label,

        "teacher_confidence":
            teacher_confidence,
    })


df_training_changes = pd.DataFrame(
    training_changes
)


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 70)
print("CORRECTED TRAINING CANDIDATE")
print("=" * 70)

print(
    "Original samples :",
    len(
        df_train_before_relabel
    )
)

print(
    "Corrected samples:",
    len(
        df_train_corrected_candidate
    )
)

print(
    "Labels changed   :",
    len(
        df_training_changes
    )
)

print()


print("BEFORE:")
print(
    df_train_before_relabel[
        "label"
    ]
    .value_counts()
    .reindex(
        LABELS,
        fill_value=0
    )
)

print()

print("AFTER:")
print(
    df_train_corrected_candidate[
        "label"
    ]
    .value_counts()
    .reindex(
        LABELS,
        fill_value=0
    )
)

print()

display(
    df_training_changes
)

print("=" * 70)


CORRECTED TRAINING CANDIDATE
Original samples : 378
Corrected samples: 378
Labels changed   : 31

BEFORE:
label
SIMPLE            64
GENERAL           65
REASONING         91
CODING_SIMPLE     41
CODING_COMPLEX    45
TRANSFORM         40
CREATIVE          32
Name: count, dtype: int64

AFTER:
label
SIMPLE            64
GENERAL           56
REASONING         71
CODING_SIMPLE     42
CODING_COMPLEX    72
TRANSFORM         40
CREATIVE          33
Name: count, dtype: int64



,prompt,old_label,new_label,teacher_confidence
0,Berikan contoh respon yang menunjukkan perilak...,GENERAL,CREATIVE,0.90
1,Bagaimana mengatur concurrency control pada da...,GENERAL,REASONING,0.90
2,Jelaskan cara mendeteksi dan memperbaiki memor...,GENERAL,CODING_COMPLEX,0.95
3,I'm building a distributed event processing pi...,GENERAL,CODING_COMPLEX,0.95
4,Saya menggunakan Django REST Framework untuk b...,GENERAL,CODING_SIMPLE,0.90
5,What are best practices for securing a Docker ...,GENERAL,REASONING,0.90
6,Can you explain how to detect and eliminate de...,GENERAL,CODING_COMPLEX,0.90
7,Saya perlu merancang arsitektur mikrolayanan u...,GENERAL,CODING_COMPLEX,0.90
8,Explain how to design a fault‑tolerant distrib...,GENERAL,CODING_COMPLEX,0.90
9,Bagaimana cara mendeteksi dan memperbaiki race...,REASONING,CODING_COMPLEX,0.95


In [40]:
# =========================================================
# CELL 29 — TRAIN CORRECTED CANDIDATE
# =========================================================

print()
print("=" * 70)
print("TRAINING CORRECTED DATASET CANDIDATE")
print("=" * 70)


(
    corrected_classifier,
    corrected_metrics,
    corrected_evaluation,
) = train_and_evaluate_router(
    df_train_corrected_candidate,
    df_validation_corrected
)


print()
print("=" * 70)
print("CORRECTED MODEL RESULT")
print("=" * 70)

print(
    "Accuracy :",
    round(
        corrected_metrics[
            "accuracy"
        ],
        4
    )
)

print(
    "Macro F1 :",
    round(
        corrected_metrics[
            "macro_f1"
        ],
        4
    )
)

print(
    "CV F1    :",
    round(
        corrected_metrics[
            "cv_macro_f1"
        ],
        4
    )
)

print(
    "Best C   :",
    corrected_metrics[
        "best_c"
    ]
)

print()

print("Per-class recall:")

for label in LABELS:

    print(
        f"{label:16} "
        f"{corrected_metrics['per_class_recall'][label]:.3f}"
    )

print("=" * 70)


TRAINING CORRECTED DATASET CANDIDATE

Generating TRAIN embeddings...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


Generating VALIDATION embeddings...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Training distribution:
SIMPLE            64
GENERAL           56
REASONING         71
CODING_SIMPLE     42
CODING_COMPLEX    72
TRANSFORM         40
CREATIVE          33
Name: count, dtype: int64

Searching best C...

CORRECTED MODEL RESULT
Accuracy : 0.8
Macro F1 : 0.7931
CV F1    : 0.6544
Best C   : 3.0

Per-class recall:
SIMPLE           0.857
GENERAL          0.571
REASONING        0.632
CODING_SIMPLE    1.000
CODING_COMPLEX   1.000
TRANSFORM        0.818
CREATIVE         0.750


In [41]:
# =========================================================
# CELL 30 — COMPARE BEFORE / AFTER CLEANING
# =========================================================

old_f1 = float(
    old_best_metrics[
        "macro_f1"
    ]
)

old_acc = float(
    old_best_metrics[
        "accuracy"
    ]
)


relabel_only_f1 = float(
    corrected_macro_f1
)

relabel_only_acc = float(
    corrected_accuracy
)


retrained_f1 = float(
    corrected_metrics[
        "macro_f1"
    ]
)

retrained_acc = float(
    corrected_metrics[
        "accuracy"
    ]
)


comparison_df = pd.DataFrame({
    "Stage": [
        "Original model + original validation",
        "Original model + corrected validation",
        "Corrected training + corrected validation",
    ],

    "Accuracy": [
        old_acc,
        relabel_only_acc,
        retrained_acc,
    ],

    "Macro F1": [
        old_f1,
        relabel_only_f1,
        retrained_f1,
    ],
})


comparison_df[
    "Accuracy"
] = (
    comparison_df[
        "Accuracy"
    ].round(4)
)


comparison_df[
    "Macro F1"
] = (
    comparison_df[
        "Macro F1"
    ].round(4)
)


display(
    comparison_df
)


print()
print(
    "Training correction improvement:",
    f"{retrained_f1 - relabel_only_f1:+.4f}"
)

,Stage,Accuracy,Macro F1
0,Original model + original validation,0.6947,0.7036
1,Original model + corrected validation,0.8211,0.8082
2,Corrected training + corrected validation,0.8000,0.7931



Training correction improvement: -0.0151


In [42]:
# =========================================================
# CELL 31 — FULL INDEPENDENT TEACHER AUDIT
# ROUTER AUTO TUNE V2
# =========================================================

AUDIT_OUTPUT_PATH = (
    f"{SAVE_DIR}/router_v2_full_audit.csv"
)

CLEAN_TRAIN_CANDIDATE_PATH = (
    f"{SAVE_DIR}/router_v2_clean_train_candidate.csv"
)


# =========================================================
# LOAD CURRENT ACTIVE TRAINING DATA
# =========================================================

if os.path.exists(
    V2_ACTIVE_DATASET_PATH
):

    df_full_audit_source = pd.read_csv(
        V2_ACTIVE_DATASET_PATH
    )

else:

    df_full_audit_source = (
        df_v2_train_baseline.copy()
    )


df_full_audit_source = (
    df_full_audit_source[
        [
            "prompt",
            "label",
        ]
    ]
    .dropna()
    .drop_duplicates(
        subset=["prompt"]
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 70)
print("FULL DATASET AUDIT")
print("=" * 70)

print(
    "Samples to audit:",
    len(
        df_full_audit_source
    )
)

print()

print(
    df_full_audit_source[
        "label"
    ]
    .value_counts()
    .reindex(
        LABELS,
        fill_value=0
    )
)

print("=" * 70)


FULL DATASET AUDIT
Samples to audit: 378

label
SIMPLE            64
GENERAL           65
REASONING         91
CODING_SIMPLE     41
CODING_COMPLEX    45
TRANSFORM         40
CREATIVE          32
Name: count, dtype: int64


In [43]:
# =========================================================
# CELL 31 — FULL INDEPENDENT TEACHER AUDIT
# ROUTER AUTO TUNE V2
# =========================================================

AUDIT_OUTPUT_PATH = (
    f"{SAVE_DIR}/router_v2_full_audit.csv"
)

CLEAN_TRAIN_CANDIDATE_PATH = (
    f"{SAVE_DIR}/router_v2_clean_train_candidate.csv"
)


# =========================================================
# LOAD CURRENT ACTIVE TRAINING DATA
# =========================================================

if os.path.exists(
    V2_ACTIVE_DATASET_PATH
):

    df_full_audit_source = pd.read_csv(
        V2_ACTIVE_DATASET_PATH
    )

else:

    df_full_audit_source = (
        df_v2_train_baseline.copy()
    )


df_full_audit_source = (
    df_full_audit_source[
        [
            "prompt",
            "label",
        ]
    ]
    .dropna()
    .drop_duplicates(
        subset=["prompt"]
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 70)
print("FULL DATASET AUDIT")
print("=" * 70)

print(
    "Samples to audit:",
    len(
        df_full_audit_source
    )
)

print()

print(
    df_full_audit_source[
        "label"
    ]
    .value_counts()
    .reindex(
        LABELS,
        fill_value=0
    )
)

print("=" * 70)


FULL DATASET AUDIT
Samples to audit: 378

label
SIMPLE            64
GENERAL           65
REASONING         91
CODING_SIMPLE     41
CODING_COMPLEX    45
TRANSFORM         40
CREATIVE          32
Name: count, dtype: int64


In [44]:
# =========================================================
# CELL 32 — TWO-STAGE TEACHER VERIFICATION
# =========================================================

def teacher_classify_once(
    prompt
):

    raw = call_teacher(
        prompt
    )

    parsed = parse_teacher_output(
        raw
    )

    return parsed


def teacher_verify_label(
    prompt,
    proposed_label
):

    proposed_definition = (
        CATEGORY_DESCRIPTIONS[
            proposed_label
        ]
    )

    verification_prompt = f"""
You are verifying a routing label.

USER PROMPT:
{prompt}

PROPOSED LABEL:
{proposed_label}

PROPOSED LABEL DEFINITION:
{proposed_definition}

Check whether the proposed label is genuinely the best label
according to the routing taxonomy.

If another routing label is better, return that other label.

Return ONLY valid JSON:

{{
  "label": "LABEL",
  "difficulty": 1,
  "confidence": 0.95
}}
"""

    raw = call_teacher(
        verification_prompt
    )

    parsed = parse_teacher_output(
        raw
    )

    return parsed

In [45]:
# =========================================================
# CELL 33 — RUN FULL AUDIT
# =========================================================

full_audit_rows = []


for _, row in tqdm(
    df_full_audit_source.iterrows(),
    total=len(
        df_full_audit_source
    ),
    desc="Full independent audit"
):

    prompt = str(
        row["prompt"]
    ).strip()

    old_label = str(
        row["label"]
    ).strip()


    # =====================================================
    # PASS 1 — CLASSIFY
    # =====================================================

    pass1 = None
    pass2 = None

    error_text = None


    try:

        pass1 = teacher_classify_once(
            prompt
        )

    except Exception as e:

        error_text = (
            "PASS1: "
            + repr(e)
        )


    # =====================================================
    # PASS 2 — VERIFY PASS 1 LABEL
    # =====================================================

    if pass1 is not None:

        try:

            pass2 = teacher_verify_label(
                prompt,
                pass1[
                    "label"
                ]
            )

        except Exception as e:

            error_text = (
                (
                    error_text + " | "
                )
                if error_text
                else ""
            ) + (
                "PASS2: "
                + repr(e)
            )


    # =====================================================
    # DECISION
    # =====================================================

    if (
        pass1 is None
        or
        pass2 is None
    ):

        final_label = None

        status = (
            "ERROR"
        )

        trusted = False


    else:

        label1 = (
            pass1[
                "label"
            ]
        )

        label2 = (
            pass2[
                "label"
            ]
        )

        conf1 = float(
            pass1[
                "confidence"
            ]
        )

        conf2 = float(
            pass2[
                "confidence"
            ]
        )


        # -----------------------------------------
        # STRONG AGREEMENT
        # -----------------------------------------

        if (
            label1 == label2
            and
            conf1 >= 0.90
            and
            conf2 >= 0.90
        ):

            final_label = label1

            status = (
                "TRUSTED"
            )

            trusted = True


        # -----------------------------------------
        # AGREEMENT BUT LOWER CONFIDENCE
        # -----------------------------------------

        elif (
            label1 == label2
            and
            conf1 >= MIN_CONFIDENCE
            and
            conf2 >= MIN_CONFIDENCE
        ):

            final_label = label1

            status = (
                "REVIEW"
            )

            trusted = False


        # -----------------------------------------
        # DISAGREEMENT
        # -----------------------------------------

        else:

            final_label = None

            status = (
                "AMBIGUOUS"
            )

            trusted = False


    # =====================================================
    # APPEND
    # =====================================================

    full_audit_rows.append({
        "prompt":
            prompt,

        "old_label":
            old_label,

        "pass1_label":
            (
                pass1["label"]
                if pass1
                else None
            ),

        "pass1_confidence":
            (
                float(
                    pass1[
                        "confidence"
                    ]
                )
                if pass1
                else None
            ),

        "pass2_label":
            (
                pass2["label"]
                if pass2
                else None
            ),

        "pass2_confidence":
            (
                float(
                    pass2[
                        "confidence"
                    ]
                )
                if pass2
                else None
            ),

        "final_label":
            final_label,

        "status":
            status,

        "trusted":
            trusted,

        "changed_from_old":
            (
                final_label is not None
                and
                final_label != old_label
            ),

        "error":
            error_text,
    })


df_full_audit = pd.DataFrame(
    full_audit_rows
)


df_full_audit.to_csv(
    AUDIT_OUTPUT_PATH,
    index=False
)


print()
print("=" * 70)
print("FULL AUDIT FINISHED")
print("=" * 70)

print(
    "Saved:",
    AUDIT_OUTPUT_PATH
)

print()

print(
    df_full_audit[
        "status"
    ]
    .value_counts()
)

print()

print(
    "Trusted relabels:",
    int(
        (
            (
                df_full_audit[
                    "trusted"
                ]
                == True
            )
            &
            (
                df_full_audit[
                    "changed_from_old"
                ]
                == True
            )
        )
        .sum()
    )
)

print("=" * 70)

Full independent audit:   0%|          | 0/378 [00:00<?, ?it/s]

  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.58s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.80s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.56s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.75s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.58s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.51s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.65s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.91s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: deepseek-v4-flash
  <- HTTP 200 (0.58s)
  -> POST https://api.deepseek.com/chat/completions
  -> model: 

In [46]:
# =========================================================
# CELL 34 — BUILD CLEAN TRAINING DATASET
# ROUTER AUTO TUNE V2
# =========================================================

CLEAN_TRAIN_PATH = (
    f"{SAVE_DIR}/router_v2_clean_train.csv"
)

REVIEW_DATA_PATH = (
    f"{SAVE_DIR}/router_v2_review_samples.csv"
)


# =========================================================
# TRUSTED ONLY
# =========================================================

df_trusted = (
    df_full_audit[
        df_full_audit[
            "status"
        ] == "TRUSTED"
    ]
    .copy()
    .reset_index(drop=True)
)


print()
print("=" * 70)
print("BUILD CLEAN TRAINING DATASET")
print("=" * 70)

print(
    "Total audited :",
    len(df_full_audit)
)

print(
    "Trusted       :",
    len(df_trusted)
)


# =========================================================
# BUILD CLEAN DATA
# =========================================================

df_clean_train = pd.DataFrame({
    "prompt":
        df_trusted[
            "prompt"
        ],

    "label":
        df_trusted[
            "final_label"
        ],
})


df_clean_train = (
    df_clean_train
    .dropna()
    .drop_duplicates(
        subset=["prompt"],
        keep="first"
    )
    .reset_index(drop=True)
)


# =========================================================
# SAFETY CHECK
# =========================================================

df_clean_train = (
    df_clean_train[
        df_clean_train[
            "label"
        ].isin(
            LABELS
        )
    ]
    .reset_index(drop=True)
)


# =========================================================
# REVIEW / AMBIGUOUS / ERROR
# =========================================================

df_review_samples = (
    df_full_audit[
        df_full_audit[
            "status"
        ] != "TRUSTED"
    ]
    .copy()
    .reset_index(drop=True)
)


# =========================================================
# SAVE
# =========================================================

df_clean_train.to_csv(
    CLEAN_TRAIN_PATH,
    index=False
)

df_review_samples.to_csv(
    REVIEW_DATA_PATH,
    index=False
)


# =========================================================
# REPORT
# =========================================================

print()
print("Clean train saved:")
print(CLEAN_TRAIN_PATH)

print()
print("Review samples saved:")
print(REVIEW_DATA_PATH)

print()
print("Clean dataset distribution:")

print(
    df_clean_train[
        "label"
    ]
    .value_counts()
    .reindex(
        LABELS,
        fill_value=0
    )
)

print()
print(
    "Review / ambiguous:",
    len(df_review_samples)
)

print("=" * 70)


BUILD CLEAN TRAINING DATASET
Total audited : 378
Trusted       : 327

Clean train saved:
/content/drive/MyDrive/router_classifier/router_v2_clean_train.csv

Review samples saved:
/content/drive/MyDrive/router_classifier/router_v2_review_samples.csv

Clean dataset distribution:
label
SIMPLE            39
GENERAL           55
REASONING         37
CODING_SIMPLE     34
CODING_COMPLEX    87
TRANSFORM         43
CREATIVE          32
Name: count, dtype: int64

Review / ambiguous: 51
